In [3]:
from pathlib import Path
import sys
import importlib

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

# Remove cached module if it exists
if "track_state" in sys.modules:
    del sys.modules["track_state"]

import track_state

print("Loaded from:")
print(track_state.__file__)

print("\nAvailable names:")
print([name for name in dir(track_state) if not name.startswith("_")])

Loaded from:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/track_state.py

Available names:
['List', 'Optional', 'TrackState', 'Tuple', 'dataclass', 'field']


In [4]:
from pathlib import Path
import sys

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

from track_state import TrackState

state = TrackState(
    canonical_id=1,
    current_track_id=101,
    position=(500, 300),
    last_frame=1,
)

state.update(
    track_id=101,
    position=(505, 302),
    frame_id=2,
)

print("TrackState test successful")
print("--------------------------------")
print("Canonical ID:", state.canonical_id)
print("Current MixSort ID:", state.current_track_id)
print("Position:", state.position)
print("Previous position:", state.previous_position)
print("Velocity:", state.velocity)
print("Speed:", state.speed())
print("Track history:", state.track_history)

TrackState test successful
--------------------------------
Canonical ID: 1
Current MixSort ID: 101
Position: (505.0, 302.0)
Previous position: (500, 300)
Velocity: (5.0, 2.0)
Speed: 5.385164807134504
Track history: [101]


In [5]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
TRACK_STITCHER_PATH = TRACKING_DIR / "track_stitcher.py"

track_stitcher_code = r'''
from typing import Dict, List, Tuple
import math

from track_state import TrackState


class TrackStitcher:
    """
    Converts temporary MixSort track IDs into persistent
    AthletiQ canonical player IDs.

    Initial matching signals:
        1. Spatial distance
        2. Motion consistency
        3. Temporal gap

    MixSort remains responsible for normal frame-to-frame tracking.
    This module only attempts to recover identities after a
    temporary track disappears and another appears.
    """

    def __init__(
        self,
        max_distance=100.0,
        max_frame_gap=30,
        velocity_weight=0.3,
    ):
        self.max_distance = float(max_distance)
        self.max_frame_gap = int(max_frame_gap)
        self.velocity_weight = float(velocity_weight)

        # canonical_id -> TrackState
        self.active_states: Dict[int, TrackState] = {}

        # MixSort temporary ID -> canonical ID
        self.track_to_canonical: Dict[int, int] = {}

        # Next canonical player ID
        self.next_canonical_id = 1

        # Statistics
        self.total_stitches = 0
        self.total_new_players = 0

    # ------------------------------------------------------------------
    # Utility
    # ------------------------------------------------------------------

    @staticmethod
    def center_from_bbox(bbox):
        x1, y1, x2, y2 = bbox

        return (
            (float(x1) + float(x2)) / 2.0,
            (float(y1) + float(y2)) / 2.0,
        )

    @staticmethod
    def euclidean_distance(p1, p2):
        return math.sqrt(
            (p1[0] - p2[0]) ** 2 +
            (p1[1] - p2[1]) ** 2
        )

    # ------------------------------------------------------------------
    # Create a new canonical player
    # ------------------------------------------------------------------

    def _create_new_player(
        self,
        track_id,
        position,
        frame_id,
        bbox,
    ):
        canonical_id = self.next_canonical_id
        self.next_canonical_id += 1

        state = TrackState(
            canonical_id=canonical_id,
            current_track_id=track_id,
            position=position,
            bbox=bbox,
            last_frame=frame_id,
            identity_confidence=1.0,
            track_history=[track_id],
        )

        self.active_states[canonical_id] = state
        self.track_to_canonical[track_id] = canonical_id

        self.total_new_players += 1

        return canonical_id

    # ------------------------------------------------------------------
    # Candidate scoring
    # ------------------------------------------------------------------

    def _candidate_score(
        self,
        state,
        position,
        frame_id,
    ):
        frame_gap = frame_id - state.last_frame

        if frame_gap <= 0:
            return None

        if frame_gap > self.max_frame_gap:
            return None

        # Predict where the player should be.
        predicted_position = (
            state.position[0] + state.velocity[0] * frame_gap,
            state.position[1] + state.velocity[1] * frame_gap,
        )

        distance = self.euclidean_distance(
            predicted_position,
            position,
        )

        if distance > self.max_distance:
            return None

        # Motion consistency.
        displacement = (
            position[0] - state.position[0],
            position[1] - state.position[1],
        )

        displacement_magnitude = math.sqrt(
            displacement[0] ** 2 +
            displacement[1] ** 2
        )

        velocity_magnitude = math.sqrt(
            state.velocity[0] ** 2 +
            state.velocity[1] ** 2
        )

        if velocity_magnitude > 1e-6 and displacement_magnitude > 1e-6:

            dot = (
                displacement[0] * state.velocity[0] +
                displacement[1] * state.velocity[1]
            )

            cosine_similarity = dot / (
                displacement_magnitude *
                velocity_magnitude
            )

            cosine_similarity = max(
                -1.0,
                min(1.0, cosine_similarity)
            )

        else:
            cosine_similarity = 1.0

        # Normalize distance.
        distance_score = max(
            0.0,
            1.0 - distance / self.max_distance
        )

        motion_score = (
            cosine_similarity + 1.0
        ) / 2.0

        score = (
            (1.0 - self.velocity_weight) * distance_score
            + self.velocity_weight * motion_score
        )

        return score

    # ------------------------------------------------------------------
    # Find best previous player
    # ------------------------------------------------------------------

    def _find_best_match(
        self,
        track_id,
        position,
        frame_id,
    ):
        best_canonical_id = None
        best_score = -1.0

        for canonical_id, state in self.active_states.items():

            # Do not match the same temporary track to itself.
            if state.current_track_id == track_id:
                continue

            score = self._candidate_score(
                state,
                position,
                frame_id,
            )

            if score is None:
                continue

            if score > best_score:
                best_score = score
                best_canonical_id = canonical_id

        return best_canonical_id, best_score

    # ------------------------------------------------------------------
    # Main update
    # ------------------------------------------------------------------

    def update(
        self,
        tracks: List[dict],
        frame_id: int,
    ):
        """
        Parameters
        ----------
        tracks:
            List returned by AthletiQMixSort.

            Example:
            {
                "track_id": 42,
                "bbox": [x1, y1, x2, y2],
                "score": 0.91
            }

        frame_id:
            Current video frame number.

        Returns
        -------
        List of dictionaries containing:
            mixsort_id
            canonical_id
            bbox
            center
            score
            stitched
            identity_confidence
        """

        frame_id = int(frame_id)

        results = []

        current_canonical_ids = set()

        # --------------------------------------------------------------
        # Process current MixSort tracks
        # --------------------------------------------------------------

        for track in tracks:

            track_id = int(track["track_id"])
            bbox = list(map(float, track["bbox"]))
            score = float(track.get("score", 1.0))

            position = self.center_from_bbox(bbox)

            # ----------------------------------------------------------
            # Existing temporary track
            # ----------------------------------------------------------

            if track_id in self.track_to_canonical:

                canonical_id = self.track_to_canonical[track_id]

                state = self.active_states[canonical_id]

                state.update(
                    track_id=track_id,
                    position=position,
                    frame_id=frame_id,
                    bbox=bbox,
                    identity_confidence=1.0,
                )

                stitched = False
                identity_confidence = 1.0

            # ----------------------------------------------------------
            # New temporary track
            # ----------------------------------------------------------

            else:

                canonical_id, match_score = self._find_best_match(
                    track_id=track_id,
                    position=position,
                    frame_id=frame_id,
                )

                if canonical_id is not None:

                    state = self.active_states[canonical_id]

                    state.update(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                        bbox=bbox,
                        identity_confidence=match_score,
                    )

                    self.track_to_canonical[track_id] = canonical_id

                    self.total_stitches += 1

                    stitched = True
                    identity_confidence = match_score

                else:

                    canonical_id = self._create_new_player(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                        bbox=bbox,
                    )

                    stitched = False
                    identity_confidence = 1.0

            current_canonical_ids.add(canonical_id)

            results.append({
                "mixsort_id": track_id,
                "canonical_id": canonical_id,
                "bbox": bbox,
                "center": position,
                "score": score,
                "stitched": stitched,
                "identity_confidence": float(identity_confidence),
            })

        # --------------------------------------------------------------
        # Mark missing canonical players
        # --------------------------------------------------------------

        for canonical_id, state in self.active_states.items():

            if canonical_id not in current_canonical_ids:
                state.mark_missing()

        # --------------------------------------------------------------
        # Remove stale mappings
        # --------------------------------------------------------------

        stale_ids = []

        for canonical_id, state in self.active_states.items():

            if state.missed_frames > self.max_frame_gap:
                stale_ids.append(canonical_id)

        for canonical_id in stale_ids:

            state = self.active_states[canonical_id]

            # Remove temporary-ID mappings belonging to this state.
            for track_id in list(self.track_to_canonical.keys()):

                if self.track_to_canonical[track_id] == canonical_id:
                    del self.track_to_canonical[track_id]

            del self.active_states[canonical_id]

        return results

    # ------------------------------------------------------------------
    # Reset
    # ------------------------------------------------------------------

    def reset(self):

        self.active_states.clear()
        self.track_to_canonical.clear()

        self.next_canonical_id = 1

        self.total_stitches = 0
        self.total_new_players = 0

    # ------------------------------------------------------------------
    # Statistics
    # ------------------------------------------------------------------

    def statistics(self):

        return {
            "active_players": len(self.active_states),
            "canonical_players_created": self.next_canonical_id - 1,
            "total_stitches": self.total_stitches,
            "total_new_players": self.total_new_players,
        }
'''

with open(TRACK_STITCHER_PATH, "w") as f:
    f.write(track_stitcher_code)

print("Created:")
print(TRACK_STITCHER_PATH)

Created:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/track_stitcher.py


In [6]:
from pathlib import Path
import sys

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

# Reload both modules
for module_name in ["track_state", "track_stitcher"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from track_stitcher import TrackStitcher


stitcher = TrackStitcher(
    max_distance=100,
    max_frame_gap=10,
    velocity_weight=0.3,
)


# ------------------------------------------------------------
# Frame 1
# Player appears with MixSort ID 101
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 101,
        "bbox": [480, 280, 520, 360],
        "score": 0.95,
    }
]

result = stitcher.update(tracks, frame_id=1)

print("Frame 1:")
print(result)


# ------------------------------------------------------------
# Frame 2
# Same MixSort ID 101 moves
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 101,
        "bbox": [485, 282, 525, 362],
        "score": 0.95,
    }
]

result = stitcher.update(tracks, frame_id=2)

print("\nFrame 2:")
print(result)


# ------------------------------------------------------------
# Frame 3
# MixSort ID SWITCH:
# 101 disappears
# New MixSort ID 202 appears nearby
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 202,
        "bbox": [490, 284, 530, 364],
        "score": 0.94,
    }
]

result = stitcher.update(tracks, frame_id=3)

print("\nFrame 3 — simulated ID switch:")
print(result)


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("\nStatistics:")
print(stitcher.statistics())

Frame 1:
[{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [480.0, 280.0, 520.0, 360.0], 'center': (500.0, 320.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]

Frame 2:
[{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [485.0, 282.0, 525.0, 362.0], 'center': (505.0, 322.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]

Frame 3 — simulated ID switch:
[{'mixsort_id': 202, 'canonical_id': 1, 'bbox': [490.0, 284.0, 530.0, 364.0], 'center': (510.0, 324.0), 'score': 0.94, 'stitched': True, 'identity_confidence': 1.0}]

Statistics:
{'active_players': 1, 'canonical_players_created': 1, 'total_stitches': 1, 'total_new_players': 1}


In [7]:
from track_stitcher import TrackStitcher

stitcher = TrackStitcher(
    max_distance=50,
    max_frame_gap=10,
    velocity_weight=0.3,
)

# ------------------------------------------------------------
# Frame 1
# Two different players
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 101,
        "bbox": [480, 280, 520, 360],
        "score": 0.95,
    },
    {
        "track_id": 102,
        "bbox": [650, 280, 690, 360],
        "score": 0.94,
    },
]

result = stitcher.update(tracks, frame_id=1)

print("Frame 1:")
for r in result:
    print(r)


# ------------------------------------------------------------
# Frame 2
# Both players continue
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 101,
        "bbox": [485, 282, 525, 362],
        "score": 0.95,
    },
    {
        "track_id": 102,
        "bbox": [645, 282, 685, 362],
        "score": 0.94,
    },
]

result = stitcher.update(tracks, frame_id=2)

print("\nFrame 2:")
for r in result:
    print(r)


# ------------------------------------------------------------
# Frame 3
# Player 101 disappears.
# A NEW ID 202 appears near Player 101.
#
# This is intentionally ambiguous.
# ------------------------------------------------------------

tracks = [
    {
        "track_id": 202,
        "bbox": [492, 284, 532, 364],
        "score": 0.93,
    },
    {
        "track_id": 102,
        "bbox": [640, 284, 680, 364],
        "score": 0.94,
    },
]

result = stitcher.update(tracks, frame_id=3)

print("\nFrame 3:")
for r in result:
    print(r)

print("\nStatistics:")
print(stitcher.statistics())

Frame 1:
{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [480.0, 280.0, 520.0, 360.0], 'center': (500.0, 320.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}
{'mixsort_id': 102, 'canonical_id': 2, 'bbox': [650.0, 280.0, 690.0, 360.0], 'center': (670.0, 320.0), 'score': 0.94, 'stitched': False, 'identity_confidence': 1.0}

Frame 2:
{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [485.0, 282.0, 525.0, 362.0], 'center': (505.0, 322.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}
{'mixsort_id': 102, 'canonical_id': 2, 'bbox': [645.0, 282.0, 685.0, 362.0], 'center': (665.0, 322.0), 'score': 0.94, 'stitched': False, 'identity_confidence': 1.0}

Frame 3:
{'mixsort_id': 202, 'canonical_id': 1, 'bbox': [492.0, 284.0, 532.0, 364.0], 'center': (512.0, 324.0), 'score': 0.93, 'stitched': True, 'identity_confidence': 0.9712172157916579}
{'mixsort_id': 102, 'canonical_id': 2, 'bbox': [640.0, 284.0, 680.0, 364.0], 'center': (660.0, 324.0), 'score': 0.94, 'stitched': Fa

In [9]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
WRAPPER_PATH = TRACKING_DIR / "mixsort_tracker.py"

text = WRAPPER_PATH.read_text()

old = '''        # ------------------------------------------------------------------
        # Add MixSort to Python path
        # ------------------------------------------------------------------
        mixsort_path = str(self.mixsort_dir)

        if mixsort_path not in sys.path:
            sys.path.insert(0, mixsort_path)
'''

new = '''        # ------------------------------------------------------------------
        # Add MixSort and MixViT to Python path
        # ------------------------------------------------------------------
        mixsort_path = str(self.mixsort_dir)
        mixvit_path = str(self.mixsort_dir / "MixViT")

        # MixSort imports the top-level `yolox` package.
        if mixsort_path not in sys.path:
            sys.path.insert(0, mixsort_path)

        # MixViT internally imports `lib.*`.
        # Therefore MixViT itself must be on sys.path.
        if mixvit_path not in sys.path:
            sys.path.insert(0, mixvit_path)
'''

if old not in text:
    raise RuntimeError(
        "Expected import-path block was not found. "
        "No changes were made."
    )

text = text.replace(old, new)

WRAPPER_PATH.write_text(text)

print("Updated:")
print(WRAPPER_PATH)
print()
print("Added paths:")
print("MixSort:", ROOT / "ai-engine" / "third_party" / "MixSort")
print("MixViT :", ROOT / "ai-engine" / "third_party" / "MixSort" / "MixViT")

Updated:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/mixsort_tracker.py

Added paths:
MixSort: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort
MixViT : /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT


In [11]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
WRAPPER_PATH = (
    ROOT
    / "ai-engine"
    / "core"
    / "tracking"
    / "mixsort_tracker.py"
)

text = WRAPPER_PATH.read_text()

# We need torch imported before patching torch.load.
old_import = """import numpy as np
import torch
import yaml
"""

new_import = """import numpy as np
import torch
import yaml

# ----------------------------------------------------------------------
# PyTorch compatibility for the trusted MixSort checkpoint
#
# PyTorch 2.6+ defaults torch.load() to weights_only=True.
# MixSort's checkpoint contains legacy Python objects such as
# AverageMeter, so the official trusted checkpoint must be loaded
# with weights_only=False.
# ----------------------------------------------------------------------

_original_torch_load = torch.load


def _mixsort_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)


torch.load = _mixsort_torch_load
"""

if old_import not in text:
    raise RuntimeError(
        "Expected torch import block was not found. "
        "No changes were made."
    )

text = text.replace(old_import, new_import, 1)

WRAPPER_PATH.write_text(text)

print("Updated MixSort wrapper:")
print(WRAPPER_PATH)
print()
print("Added trusted-checkpoint torch.load compatibility.")

Updated MixSort wrapper:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/mixsort_tracker.py

Added trusted-checkpoint torch.load compatibility.


In [12]:
import sys
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

# Remove cached wrapper
for name in list(sys.modules):
    if name == "mixsort_tracker":
        del sys.modules[name]

from mixsort_tracker import AthletiQMixSort

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=25,
)

print("\n✓ AthletiQMixSort initialized successfully")
print(tracker.get_config())

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.

✓ AthletiQMixSort initialized successfully
{'track_thresh': 0.6, 'track_buffer': 30, 'match_thresh': 0.9, 'iou_thresh': 0.3, 'alpha': 0.8, 'frame_rate': 25}


In [13]:
from pathlib import Path
import sys
import gc
import time

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
MIXSORT_DIR = ROOT / "ai-engine" / "third_party" / "MixSort"

SEQ_DIR = (
    ROOT
    / "datasets"
    / "football"
    / "benchmarks"
    / "sportsmot"
    / "extracted"
    / "val"
    / "v_2QhNRucNC7E_c017"
)

MODEL_PATH = ROOT / "ai-engine" / "models" / "football_player_detector.pt"

# Output directory
OUTPUT_DIR = ROOT / "outputs" / "tracking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_LOG_PATH = OUTPUT_DIR / "c017_track_stitch_events.csv"

# ============================================================
# IMPORTS
# ============================================================

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

# Reload our modules so notebook cache doesn't cause problems
for module_name in ["track_state", "track_stitcher", "mixsort_tracker"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from track_stitcher import TrackStitcher
from mixsort_tracker import AthletiQMixSort

# ============================================================
# CONFIGURATION
# ============================================================

DETECTOR_CONF = 0.40
DETECTOR_IOU = 0.40

FPS = 25

# Current experimental stitcher parameters
STITCH_MAX_DISTANCE = 100.0
STITCH_MAX_FRAME_GAP = 30
STITCH_VELOCITY_WEIGHT = 0.30

print("=" * 80)
print("REAL c017 TRACK STITCHING INSPECTION")
print("=" * 80)

print("\nSequence:")
print(SEQ_DIR)

print("\nDetector:")
print(f"  confidence = {DETECTOR_CONF}")
print(f"  NMS IoU    = {DETECTOR_IOU}")

print("\nStitcher:")
print(f"  max_distance      = {STITCH_MAX_DISTANCE}")
print(f"  max_frame_gap     = {STITCH_MAX_FRAME_GAP}")
print(f"  velocity_weight   = {STITCH_VELOCITY_WEIGHT}")

# ============================================================
# VERIFY SEQUENCE
# ============================================================

IMG_DIR = SEQ_DIR / "img1"

frame_paths = sorted(IMG_DIR.glob("*.jpg"))

if not frame_paths:
    raise FileNotFoundError(f"No frames found in {IMG_DIR}")

print(f"\nFrames found: {len(frame_paths)}")

# ============================================================
# LOAD DETECTOR
# ============================================================

print("\nLoading player detector...")

detector = YOLO(str(MODEL_PATH))

print("Detector loaded.")

# ============================================================
# CREATE TRACKER + STITCHER
# ============================================================

print("\nInitializing MixSort...")

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=FPS,
)

print("MixSort ready.")

stitcher = TrackStitcher(
    max_distance=STITCH_MAX_DISTANCE,
    max_frame_gap=STITCH_MAX_FRAME_GAP,
    velocity_weight=STITCH_VELOCITY_WEIGHT,
)

print("TrackStitcher ready.")

# ============================================================
# PROCESS SEQUENCE
# ============================================================

all_results = []
stitch_events = []

start_time = time.time()

for frame_index, frame_path in enumerate(frame_paths, start=1):

    frame = cv2.imread(str(frame_path))

    if frame is None:
        print(f"WARNING: could not read {frame_path}")
        continue

    # --------------------------------------------------------
    # Player detection
    # --------------------------------------------------------

    result = detector.predict(
        frame,
        conf=DETECTOR_CONF,
        iou=DETECTOR_IOU,
        device=0,
        verbose=False,
    )[0]

    if result.boxes is not None and len(result.boxes) > 0:

        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()

        detections = np.column_stack(
            [boxes, scores]
        ).astype(np.float32)

    else:

        detections = np.empty(
            (0, 5),
            dtype=np.float32
        )

    # --------------------------------------------------------
    # MixSort
    # --------------------------------------------------------

    mixsort_tracks = tracker.update(
        detections,
        frame,
    )

    # --------------------------------------------------------
    # Track stitching
    # --------------------------------------------------------

    canonical_tracks = stitcher.update(
        mixsort_tracks,
        frame_id=frame_index,
    )

    # --------------------------------------------------------
    # Store all current tracks
    # --------------------------------------------------------

    for track in canonical_tracks:

        all_results.append({
            "frame_id": frame_index,
            "mixsort_id": track["mixsort_id"],
            "canonical_id": track["canonical_id"],
            "x1": track["bbox"][0],
            "y1": track["bbox"][1],
            "x2": track["bbox"][2],
            "y2": track["bbox"][3],
            "center_x": track["center"][0],
            "center_y": track["center"][1],
            "score": track["score"],
            "stitched": track["stitched"],
            "identity_confidence": track["identity_confidence"],
        })

    # --------------------------------------------------------
    # Record stitching events
    # --------------------------------------------------------

    for track in canonical_tracks:

        if track["stitched"]:

            state = stitcher.active_states[
                track["canonical_id"]
            ]

            stitch_events.append({
                "frame_id": frame_index,
                "mixsort_new_id": track["mixsort_id"],
                "canonical_id": track["canonical_id"],
                "identity_confidence": track["identity_confidence"],
                "center_x": track["center"][0],
                "center_y": track["center"][1],
                "velocity_x": state.velocity[0],
                "velocity_y": state.velocity[1],
                "speed_pixels_per_frame": state.speed(),
                "track_history": str(state.track_history),
            })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if frame_index % 50 == 0:

        elapsed = time.time() - start_time
        fps = frame_index / elapsed

        print(
            f"Frame {frame_index:4d}/{len(frame_paths)} | "
            f"MixSort tracks: {len(mixsort_tracks):2d} | "
            f"Canonical active: {len(canonical_tracks):2d} | "
            f"Stitches: {len(stitch_events):3d} | "
            f"FPS: {fps:.2f}"
        )

# ============================================================
# DATAFRAMES
# ============================================================

results_df = pd.DataFrame(all_results)
events_df = pd.DataFrame(stitch_events)

# ============================================================
# SAVE EVENT LOG
# ============================================================

events_df.to_csv(
    EVENT_LOG_PATH,
    index=False
)

# ============================================================
# FINAL SUMMARY
# ============================================================

elapsed = time.time() - start_time
overall_fps = len(frame_paths) / elapsed

print("\n" + "=" * 80)
print("STITCHING INSPECTION COMPLETE")
print("=" * 80)

print(f"\nFrames processed:       {len(frame_paths)}")
print(f"Total tracked rows:     {len(results_df)}")
print(f"Total stitch events:    {len(events_df)}")
print(f"Canonical players:      {stitcher.next_canonical_id - 1}")
print(f"Overall processing FPS: {overall_fps:.2f}")

print("\nStitcher statistics:")
print(stitcher.statistics())

print("\nEvent log saved to:")
print(EVENT_LOG_PATH)

# ============================================================
# SHOW STITCH EVENTS
# ============================================================

if len(events_df) > 0:

    print("\n" + "=" * 80)
    print("FIRST STITCHING EVENTS")
    print("=" * 80)

    display(
        events_df.head(30)
    )

else:

    print("\nNo stitching events detected.")

# ============================================================
# STITCH COUNT PER CANONICAL PLAYER
# ============================================================

if len(events_df) > 0:

    stitch_summary = (
        events_df
        .groupby("canonical_id")
        .agg(
            stitch_count=("canonical_id", "size"),
            first_stitch_frame=("frame_id", "min"),
            last_stitch_frame=("frame_id", "max"),
            min_confidence=("identity_confidence", "min"),
            mean_confidence=("identity_confidence", "mean"),
            mixsort_ids=("mixsort_new_id", lambda x: list(x)),
        )
        .sort_values(
            "stitch_count",
            ascending=False
        )
    )

    print("\n" + "=" * 80)
    print("STITCHES BY CANONICAL PLAYER")
    print("=" * 80)

    display(stitch_summary)

REAL c017 TRACK STITCHING INSPECTION

Sequence:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017

Detector:
  confidence = 0.4
  NMS IoU    = 0.4

Stitcher:
  max_distance      = 100.0
  max_frame_gap     = 30
  velocity_weight   = 0.3

Frames found: 450

Loading player detector...
Detector loaded.

Initializing MixSort...
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MixSort ready.
TrackStitcher ready.
Frame   50/450 | MixSort tracks: 19 | Canonical active: 19 | Stitches:   0 | FPS: 2.92
Frame  100/450 | MixSort tracks: 16 | Canonical active: 16 | Stitches:   0 | FPS: 3.39
Frame  150/450 | MixSort tracks: 15 | Canonical active: 15 | Stitches:   0 | FPS: 3.64
Frame  200/450 | MixSort tracks: 16 | Canonical active: 16 | Stitches:   0 | FPS: 3.60
Frame  250/450 | MixSort tracks: 15 | Canonical active: 15 | Stitches:   0 | FP

,frame_id,mixsort_new_id,canonical_id,identity_confidence,center_x,center_y,velocity_x,velocity_y,speed_pixels_per_frame,track_history
0,351,39,22,0.771495,753.645756,342.835077,33.539763,-21.317832,39.741234,"[30, 39]"
1,371,41,4,0.645254,15.742959,435.538704,7.460310,-30.606238,31.502350,"[4, 41]"
2,377,62,24,0.885360,1035.047240,338.586431,35.976293,-0.321909,35.977733,"[33, 62]"
3,385,116,22,0.563468,1253.296343,295.354238,-99.498170,5.121747,99.629906,"[30, 39, 116, 39]"
4,389,142,17,0.619772,718.591727,332.965770,227.966884,60.636077,235.893268,"[17, 142]"
5,391,160,8,0.772858,1168.049961,254.168928,176.768228,8.110499,176.954194,"[8, 160]"
6,399,188,23,0.616113,917.929931,266.840260,39.914645,37.415941,54.709520,"[31, 188]"



STITCHES BY CANONICAL PLAYER


,stitch_count,first_stitch_frame,last_stitch_frame,min_confidence,mean_confidence,mixsort_ids
canonical_id,,,,,,
22,2,351,385,0.563468,0.667482,"[39, 116]"
4,1,371,371,0.645254,0.645254,[41]
8,1,391,391,0.772858,0.772858,[160]
17,1,389,389,0.619772,0.619772,[142]
23,1,399,399,0.616113,0.616113,[188]
24,1,377,377,0.885360,0.885360,[62]


In [14]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
STITCHER_PATH = TRACKING_DIR / "track_stitcher.py"

new_code = r'''
from typing import Dict, List
import math

from track_state import TrackState


class TrackStitcher:
    """
    Identity recovery layer on top of MixSort.

    MixSort IDs are temporary.
    canonical_id represents the persistent AthletiQ player identity.

    Lifecycle:

        ACTIVE
          |
          | disappears
          v
        LOST
          |
          | new MixSort ID matches
          v
        ACTIVE

    A new MixSort ID is NEVER matched against currently active
    players. It is only matched against recently lost players.
    """

    def __init__(
        self,
        max_distance=100.0,
        max_frame_gap=30,
        velocity_weight=0.30,
        min_stitch_confidence=0.70,
    ):

        self.max_distance = float(max_distance)
        self.max_frame_gap = int(max_frame_gap)
        self.velocity_weight = float(velocity_weight)
        self.min_stitch_confidence = float(min_stitch_confidence)

        # All canonical player states
        self.states: Dict[int, TrackState] = {}

        # Current MixSort ID -> canonical ID
        self.track_to_canonical: Dict[int, int] = {}

        # Canonical IDs currently visible
        self.active_ids = set()

        # Canonical IDs temporarily missing
        self.lost_ids = set()

        self.next_canonical_id = 1

        self.total_stitches = 0
        self.total_new_players = 0

    # ================================================================
    # Utility
    # ================================================================

    @staticmethod
    def center_from_bbox(bbox):

        x1, y1, x2, y2 = bbox

        return (
            (float(x1) + float(x2)) / 2.0,
            (float(y1) + float(y2)) / 2.0,
        )

    @staticmethod
    def euclidean_distance(p1, p2):

        return math.sqrt(
            (p1[0] - p2[0]) ** 2 +
            (p1[1] - p2[1]) ** 2
        )

    # ================================================================
    # Create new canonical player
    # ================================================================

    def _create_new_player(
        self,
        track_id,
        position,
        frame_id,
        bbox,
    ):

        canonical_id = self.next_canonical_id

        self.next_canonical_id += 1

        state = TrackState(
            canonical_id=canonical_id,
            current_track_id=track_id,
            position=position,
            bbox=bbox,
            last_frame=frame_id,
            identity_confidence=1.0,
            track_history=[track_id],
        )

        self.states[canonical_id] = state

        self.track_to_canonical[track_id] = canonical_id

        self.active_ids.add(canonical_id)

        self.total_new_players += 1

        return canonical_id

    # ================================================================
    # Candidate scoring
    # ================================================================

    def _candidate_score(
        self,
        state,
        position,
        frame_id,
    ):

        frame_gap = frame_id - state.last_frame

        if frame_gap <= 0:
            return None

        if frame_gap > self.max_frame_gap:
            return None

        # ------------------------------------------------------------
        # Predict position from previous motion
        # ------------------------------------------------------------

        predicted_position = (
            state.position[0] +
            state.velocity[0] * frame_gap,

            state.position[1] +
            state.velocity[1] * frame_gap,
        )

        distance = self.euclidean_distance(
            predicted_position,
            position,
        )

        if distance > self.max_distance:
            return None

        # ------------------------------------------------------------
        # Distance score
        # ------------------------------------------------------------

        distance_score = max(
            0.0,
            1.0 - distance / self.max_distance
        )

        # ------------------------------------------------------------
        # Motion direction consistency
        # ------------------------------------------------------------

        displacement = (
            position[0] - state.position[0],
            position[1] - state.position[1],
        )

        displacement_magnitude = math.sqrt(
            displacement[0] ** 2 +
            displacement[1] ** 2
        )

        velocity_magnitude = math.sqrt(
            state.velocity[0] ** 2 +
            state.velocity[1] ** 2
        )

        if (
            velocity_magnitude > 1e-6
            and displacement_magnitude > 1e-6
        ):

            dot = (
                displacement[0] * state.velocity[0]
                +
                displacement[1] * state.velocity[1]
            )

            cosine_similarity = dot / (
                displacement_magnitude *
                velocity_magnitude
            )

            cosine_similarity = max(
                -1.0,
                min(1.0, cosine_similarity)
            )

        else:

            cosine_similarity = 1.0

        motion_score = (
            cosine_similarity + 1.0
        ) / 2.0

        # ------------------------------------------------------------
        # Final score
        # ------------------------------------------------------------

        score = (
            (1.0 - self.velocity_weight) * distance_score
            +
            self.velocity_weight * motion_score
        )

        return score

    # ================================================================
    # Find best LOST player
    # ================================================================

    def _find_best_lost_match(
        self,
        track_id,
        position,
        frame_id,
    ):

        best_canonical_id = None
        best_score = -1.0

        for canonical_id in list(self.lost_ids):

            state = self.states[canonical_id]

            score = self._candidate_score(
                state,
                position,
                frame_id,
            )

            if score is None:
                continue

            if score > best_score:

                best_score = score
                best_canonical_id = canonical_id

        # ------------------------------------------------------------
        # Conservative acceptance threshold
        # ------------------------------------------------------------

        if (
            best_canonical_id is None
            or best_score < self.min_stitch_confidence
        ):

            return None, best_score

        return best_canonical_id, best_score

    # ================================================================
    # Update
    # ================================================================

    def update(
        self,
        tracks: List[dict],
        frame_id: int,
    ):

        frame_id = int(frame_id)

        results = []

        current_ids = set()

        # Track IDs appearing in this frame
        current_mixsort_ids = {
            int(track["track_id"])
            for track in tracks
        }

        # ============================================================
        # Process current MixSort tracks
        # ============================================================

        for track in tracks:

            track_id = int(track["track_id"])

            bbox = list(
                map(float, track["bbox"])
            )

            score = float(
                track.get("score", 1.0)
            )

            position = self.center_from_bbox(bbox)

            # --------------------------------------------------------
            # Existing MixSort ID
            # --------------------------------------------------------

            if track_id in self.track_to_canonical:

                canonical_id = (
                    self.track_to_canonical[track_id]
                )

                state = self.states[canonical_id]

                state.update(
                    track_id=track_id,
                    position=position,
                    frame_id=frame_id,
                    bbox=bbox,
                    identity_confidence=1.0,
                )

                self.active_ids.add(
                    canonical_id
                )

                self.lost_ids.discard(
                    canonical_id
                )

                stitched = False
                identity_confidence = 1.0

            # --------------------------------------------------------
            # New MixSort ID
            # --------------------------------------------------------

            else:

                canonical_id, match_score = (
                    self._find_best_lost_match(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                    )
                )

                # ----------------------------------------------------
                # Successful recovery
                # ----------------------------------------------------

                if canonical_id is not None:

                    state = self.states[
                        canonical_id
                    ]

                    state.update(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                        bbox=bbox,
                        identity_confidence=match_score,
                    )

                    self.track_to_canonical[
                        track_id
                    ] = canonical_id

                    self.active_ids.add(
                        canonical_id
                    )

                    self.lost_ids.discard(
                        canonical_id
                    )

                    self.total_stitches += 1

                    stitched = True
                    identity_confidence = match_score

                # ----------------------------------------------------
                # No valid lost-track match
                # ----------------------------------------------------

                else:

                    canonical_id = (
                        self._create_new_player(
                            track_id=track_id,
                            position=position,
                            frame_id=frame_id,
                            bbox=bbox,
                        )
                    )

                    stitched = False
                    identity_confidence = 1.0

            current_ids.add(
                canonical_id
            )

            results.append({
                "mixsort_id": track_id,
                "canonical_id": canonical_id,
                "bbox": bbox,
                "center": position,
                "score": score,
                "stitched": stitched,
                "identity_confidence": float(
                    identity_confidence
                ),
            })

        # ============================================================
        # Determine which active players disappeared
        # ============================================================

        previously_active = set(
            self.active_ids
        )

        disappeared = (
            previously_active - current_ids
        )

        for canonical_id in disappeared:

            if canonical_id not in self.states:
                continue

            self.states[
                canonical_id
            ].mark_missing()

            self.active_ids.discard(
                canonical_id
            )

            self.lost_ids.add(
                canonical_id
            )

        # ============================================================
        # Remove stale LOST players
        # ============================================================

        stale_ids = []

        for canonical_id in list(
            self.lost_ids
        ):

            state = self.states[
                canonical_id
            ]

            if (
                state.missed_frames
                > self.max_frame_gap
            ):

                stale_ids.append(
                    canonical_id
                )

        for canonical_id in stale_ids:

            self.lost_ids.discard(
                canonical_id
            )

            self.states.pop(
                canonical_id,
                None
            )

        return results

    # ================================================================
    # Reset
    # ================================================================

    def reset(self):

        self.states.clear()

        self.track_to_canonical.clear()

        self.active_ids.clear()

        self.lost_ids.clear()

        self.next_canonical_id = 1

        self.total_stitches = 0

        self.total_new_players = 0

    # ================================================================
    # Statistics
    # ================================================================

    def statistics(self):

        return {
            "active_players": len(
                self.active_ids
            ),

            "lost_players": len(
                self.lost_ids
            ),

            "canonical_players_created": (
                self.next_canonical_id - 1
            ),

            "total_stitches": (
                self.total_stitches
            ),

            "total_new_players": (
                self.total_new_players
            ),
        }
'''

TRACKING_DIR.mkdir(parents=True, exist_ok=True)

STITCHER_PATH.write_text(new_code)

print("Updated:")
print(STITCHER_PATH)
print()
print("New safeguards:")
print("  ✓ ACTIVE / LOST states")
print("  ✓ New IDs match LOST tracks only")
print("  ✓ Maximum frame gap")
print("  ✓ Maximum spatial distance")
print("  ✓ Motion consistency")
print("  ✓ Minimum stitch confidence")

Updated:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/track_stitcher.py

New safeguards:
  ✓ ACTIVE / LOST states
  ✓ New IDs match LOST tracks only
  ✓ Maximum frame gap
  ✓ Maximum spatial distance
  ✓ Motion consistency
  ✓ Minimum stitch confidence


In [15]:
import sys

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

if "track_stitcher" in sys.modules:
    del sys.modules["track_stitcher"]

from track_stitcher import TrackStitcher

stitcher = TrackStitcher(
    max_distance=100,
    max_frame_gap=10,
    velocity_weight=0.3,
    min_stitch_confidence=0.70,
)

# Frame 1
result = stitcher.update(
    [{
        "track_id": 101,
        "bbox": [480, 280, 520, 360],
        "score": 0.95,
    }],
    frame_id=1,
)

print("Frame 1:", result)

# Frame 2
result = stitcher.update(
    [{
        "track_id": 101,
        "bbox": [485, 282, 525, 362],
        "score": 0.95,
    }],
    frame_id=2,
)

print("Frame 2:", result)

# Frame 3 — ID switch
result = stitcher.update(
    [{
        "track_id": 202,
        "bbox": [490, 284, 530, 364],
        "score": 0.94,
    }],
    frame_id=3,
)

print("Frame 3:", result)

print("\nStatistics:")
print(stitcher.statistics())

Frame 1: [{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [480.0, 280.0, 520.0, 360.0], 'center': (500.0, 320.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]
Frame 2: [{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [485.0, 282.0, 525.0, 362.0], 'center': (505.0, 322.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]
Frame 3: [{'mixsort_id': 202, 'canonical_id': 2, 'bbox': [490.0, 284.0, 530.0, 364.0], 'center': (510.0, 324.0), 'score': 0.94, 'stitched': False, 'identity_confidence': 1.0}]

Statistics:
{'active_players': 1, 'lost_players': 1, 'canonical_players_created': 2, 'total_stitches': 0, 'total_new_players': 2}


In [16]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
STITCHER_PATH = TRACKING_DIR / "track_stitcher.py"

text = STITCHER_PATH.read_text()

# Replace the entire update() method with the corrected version.
start = text.index("    def update(\n")
end = text.index("    # ================================================================\n    # Reset", start)

new_update = r'''    def update(
        self,
        tracks: List[dict],
        frame_id: int,
    ):

        frame_id = int(frame_id)

        results = []

        current_ids = set()

        # ============================================================
        # IMPORTANT:
        # Determine which existing MixSort IDs disappeared BEFORE
        # trying to match new IDs.
        # ============================================================

        current_mixsort_ids = {
            int(track["track_id"])
            for track in tracks
        }

        previously_active = set(
            self.active_ids
        )

        # Canonical players whose current MixSort ID is no longer
        # present in this frame.
        disappeared = set()

        for canonical_id in previously_active:

            state = self.states.get(canonical_id)

            if state is None:
                continue

            if state.current_track_id not in current_mixsort_ids:

                disappeared.add(canonical_id)

        # Move disappeared players to LOST before matching new IDs.
        for canonical_id in disappeared:

            state = self.states[canonical_id]

            state.mark_missing()

            self.active_ids.discard(
                canonical_id
            )

            self.lost_ids.add(
                canonical_id
            )

        # ============================================================
        # Process current MixSort tracks
        # ============================================================

        for track in tracks:

            track_id = int(
                track["track_id"]
            )

            bbox = list(
                map(float, track["bbox"])
            )

            score = float(
                track.get("score", 1.0)
            )

            position = self.center_from_bbox(
                bbox
            )

            # --------------------------------------------------------
            # Existing MixSort ID
            # --------------------------------------------------------

            if track_id in self.track_to_canonical:

                canonical_id = (
                    self.track_to_canonical[
                        track_id
                    ]
                )

                # The mapping may belong to a stale state.
                if canonical_id not in self.states:

                    del self.track_to_canonical[
                        track_id
                    ]

                    canonical_id = None

                else:

                    state = self.states[
                        canonical_id
                    ]

                    state.update(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                        bbox=bbox,
                        identity_confidence=1.0,
                    )

                    self.active_ids.add(
                        canonical_id
                    )

                    self.lost_ids.discard(
                        canonical_id
                    )

                    stitched = False
                    identity_confidence = 1.0

            else:

                canonical_id = None

            # --------------------------------------------------------
            # New MixSort ID
            # --------------------------------------------------------

            if canonical_id is None:

                canonical_id, match_score = (
                    self._find_best_lost_match(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                    )
                )

                # ----------------------------------------------------
                # Successful identity recovery
                # ----------------------------------------------------

                if canonical_id is not None:

                    state = self.states[
                        canonical_id
                    ]

                    old_track_id = (
                        state.current_track_id
                    )

                    state.update(
                        track_id=track_id,
                        position=position,
                        frame_id=frame_id,
                        bbox=bbox,
                        identity_confidence=match_score,
                    )

                    self.track_to_canonical[
                        track_id
                    ] = canonical_id

                    self.active_ids.add(
                        canonical_id
                    )

                    self.lost_ids.discard(
                        canonical_id
                    )

                    self.total_stitches += 1

                    stitched = True
                    identity_confidence = match_score

                # ----------------------------------------------------
                # No valid LOST match
                # ----------------------------------------------------

                else:

                    canonical_id = (
                        self._create_new_player(
                            track_id=track_id,
                            position=position,
                            frame_id=frame_id,
                            bbox=bbox,
                        )
                    )

                    stitched = False
                    identity_confidence = 1.0

            current_ids.add(
                canonical_id
            )

            results.append({
                "mixsort_id": track_id,
                "canonical_id": canonical_id,
                "bbox": bbox,
                "center": position,
                "score": score,
                "stitched": stitched,
                "identity_confidence": float(
                    identity_confidence
                ),
            })

        # ============================================================
        # Remove stale LOST players
        # ============================================================

        stale_ids = []

        for canonical_id in list(
            self.lost_ids
        ):

            state = self.states[
                canonical_id
            ]

            if (
                state.missed_frames
                > self.max_frame_gap
            ):

                stale_ids.append(
                    canonical_id
                )

        for canonical_id in stale_ids:

            self.lost_ids.discard(
                canonical_id
            )

            state = self.states.pop(
                canonical_id,
                None
            )

            if state is not None:

                # Remove mappings associated with this state.
                for track_id in list(
                    self.track_to_canonical.keys()
                ):

                    if (
                        self.track_to_canonical[
                            track_id
                        ]
                        == canonical_id
                    ):

                        del self.track_to_canonical[
                            track_id
                        ]

        return results

'''

STITCHER_PATH.write_text(
    text[:start] +
    new_update +
    text[end:]
)

print("Updated TrackStitcher:")
print(STITCHER_PATH)
print()
print("Fixed update order:")
print("  1. Detect disappeared players")
print("  2. Move them to LOST")
print("  3. Process new MixSort IDs")
print("  4. Match new IDs against LOST players")

Updated TrackStitcher:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/track_stitcher.py

Fixed update order:
  1. Detect disappeared players
  2. Move them to LOST
  3. Process new MixSort IDs
  4. Match new IDs against LOST players


In [17]:
import sys

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

if "track_stitcher" in sys.modules:
    del sys.modules["track_stitcher"]

from track_stitcher import TrackStitcher

stitcher = TrackStitcher(
    max_distance=100,
    max_frame_gap=10,
    velocity_weight=0.3,
    min_stitch_confidence=0.70,
)

# Frame 1
print(
    "Frame 1:",
    stitcher.update(
        [{
            "track_id": 101,
            "bbox": [480, 280, 520, 360],
            "score": 0.95,
        }],
        frame_id=1,
    )
)

# Frame 2
print(
    "Frame 2:",
    stitcher.update(
        [{
            "track_id": 101,
            "bbox": [485, 282, 525, 362],
            "score": 0.95,
        }],
        frame_id=2,
    )
)

# Frame 3 — ID switch
print(
    "Frame 3:",
    stitcher.update(
        [{
            "track_id": 202,
            "bbox": [490, 284, 530, 364],
            "score": 0.94,
        }],
        frame_id=3,
    )
)

print("\nStatistics:")
print(stitcher.statistics())

Frame 1: [{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [480.0, 280.0, 520.0, 360.0], 'center': (500.0, 320.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]
Frame 2: [{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [485.0, 282.0, 525.0, 362.0], 'center': (505.0, 322.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]
Frame 3: [{'mixsort_id': 202, 'canonical_id': 1, 'bbox': [490.0, 284.0, 530.0, 364.0], 'center': (510.0, 324.0), 'score': 0.94, 'stitched': True, 'identity_confidence': 1.0}]

Statistics:
{'active_players': 1, 'lost_players': 0, 'canonical_players_created': 1, 'total_stitches': 1, 'total_new_players': 1}


In [18]:
from pathlib import Path
import sys
import gc
import time

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
MIXSORT_DIR = ROOT / "ai-engine" / "third_party" / "MixSort"

SEQ_DIR = (
    ROOT
    / "datasets"
    / "football"
    / "benchmarks"
    / "sportsmot"
    / "extracted"
    / "val"
    / "v_2QhNRucNC7E_c017"
)

MODEL_PATH = (
    ROOT
    / "ai-engine"
    / "models"
    / "football_player_detector.pt"
)

OUTPUT_DIR = ROOT / "outputs" / "tracking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_LOG_PATH = (
    OUTPUT_DIR / "c017_track_stitch_events_v2.csv"
)

# ============================================================
# CONFIGURATION
# ============================================================

DETECTOR_CONF = 0.40
DETECTOR_IOU = 0.40

FPS = 25

# Improved TrackStitcher configuration
STITCH_MAX_DISTANCE = 100.0
STITCH_MAX_FRAME_GAP = 30
STITCH_VELOCITY_WEIGHT = 0.30
MIN_STITCH_CONFIDENCE = 0.70

print("=" * 80)
print("REAL c017 TRACK STITCHING INSPECTION — V2")
print("=" * 80)

print("\nSequence:")
print(SEQ_DIR)

print("\nDetector:")
print(f"  confidence = {DETECTOR_CONF}")
print(f"  NMS IoU    = {DETECTOR_IOU}")

print("\nTrackStitcher V2:")
print(f"  max_distance       = {STITCH_MAX_DISTANCE}")
print(f"  max_frame_gap      = {STITCH_MAX_FRAME_GAP}")
print(f"  velocity_weight    = {STITCH_VELOCITY_WEIGHT}")
print(f"  min_stitch_conf.   = {MIN_STITCH_CONFIDENCE}")

# ============================================================
# VERIFY DATA
# ============================================================

IMG_DIR = SEQ_DIR / "img1"
frame_paths = sorted(IMG_DIR.glob("*.jpg"))

if not frame_paths:
    raise FileNotFoundError(
        f"No frames found in {IMG_DIR}"
    )

print(f"\nFrames found: {len(frame_paths)}")

# ============================================================
# IMPORT OUR MODULES
# ============================================================

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

# Clear cached modules so the latest files are used
for module_name in [
    "track_state",
    "track_stitcher",
    "mixsort_tracker",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from track_stitcher import TrackStitcher
from mixsort_tracker import AthletiQMixSort

# ============================================================
# LOAD DETECTOR
# ============================================================

print("\nLoading player detector...")

detector = YOLO(str(MODEL_PATH))

print("Detector loaded.")

# ============================================================
# INITIALIZE MIXSORT
# ============================================================

print("\nInitializing MixSort...")

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=FPS,
)

print("MixSort ready.")

# ============================================================
# INITIALIZE TRACK STITCHER
# ============================================================

stitcher = TrackStitcher(
    max_distance=STITCH_MAX_DISTANCE,
    max_frame_gap=STITCH_MAX_FRAME_GAP,
    velocity_weight=STITCH_VELOCITY_WEIGHT,
    min_stitch_confidence=MIN_STITCH_CONFIDENCE,
)

print("TrackStitcher V2 ready.")

# ============================================================
# STORAGE
# ============================================================

all_results = []
stitch_events = []

start_time = time.time()

# ============================================================
# PROCESS ALL 450 FRAMES
# ============================================================

for frame_index, frame_path in enumerate(
    frame_paths,
    start=1
):

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        print(
            f"WARNING: could not read {frame_path}"
        )
        continue

    # --------------------------------------------------------
    # PLAYER DETECTION
    # --------------------------------------------------------

    result = detector.predict(
        frame,
        conf=DETECTOR_CONF,
        iou=DETECTOR_IOU,
        device=0,
        verbose=False,
    )[0]

    if (
        result.boxes is not None
        and len(result.boxes) > 0
    ):

        boxes = (
            result.boxes
            .xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result.boxes
            .conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [boxes, scores]
        ).astype(np.float32)

    else:

        detections = np.empty(
            (0, 5),
            dtype=np.float32
        )

    # --------------------------------------------------------
    # MIXSORT
    # --------------------------------------------------------

    mixsort_tracks = tracker.update(
        detections,
        frame,
    )

    # --------------------------------------------------------
    # TRACK STITCHING V2
    # --------------------------------------------------------

    canonical_tracks = stitcher.update(
        mixsort_tracks,
        frame_id=frame_index,
    )

    # --------------------------------------------------------
    # SAVE ALL TRACK RESULTS
    # --------------------------------------------------------

    for track in canonical_tracks:

        all_results.append({
            "frame_id": frame_index,
            "mixsort_id": track["mixsort_id"],
            "canonical_id": track["canonical_id"],
            "x1": track["bbox"][0],
            "y1": track["bbox"][1],
            "x2": track["bbox"][2],
            "y2": track["bbox"][3],
            "center_x": track["center"][0],
            "center_y": track["center"][1],
            "score": track["score"],
            "stitched": track["stitched"],
            "identity_confidence": (
                track["identity_confidence"]
            ),
        })

    # --------------------------------------------------------
    # RECORD STITCH EVENTS
    # --------------------------------------------------------

    for track in canonical_tracks:

        if not track["stitched"]:
            continue

        canonical_id = track["canonical_id"]

        state = stitcher.states[
            canonical_id
        ]

        stitch_events.append({
            "frame_id": frame_index,
            "mixsort_new_id": track["mixsort_id"],
            "canonical_id": canonical_id,
            "identity_confidence": (
                track["identity_confidence"]
            ),
            "center_x": track["center"][0],
            "center_y": track["center"][1],
            "velocity_x": state.velocity[0],
            "velocity_y": state.velocity[1],
            "speed_pixels_per_frame": state.speed(),
            "track_history": str(
                state.track_history
            ),
        })

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if frame_index % 50 == 0:

        elapsed = time.time() - start_time
        current_fps = frame_index / elapsed

        print(
            f"Frame {frame_index:4d}/{len(frame_paths)} | "
            f"MixSort tracks: {len(mixsort_tracks):2d} | "
            f"Canonical active: {len(canonical_tracks):2d} | "
            f"Lost: {len(stitcher.lost_ids):2d} | "
            f"Stitches: {len(stitch_events):3d} | "
            f"FPS: {current_fps:.2f}"
        )

# ============================================================
# DATAFRAMES
# ============================================================

results_df = pd.DataFrame(
    all_results
)

events_df = pd.DataFrame(
    stitch_events
)

# ============================================================
# SAVE FULL TRACK DATA
# ============================================================

TRACK_LOG_PATH = (
    OUTPUT_DIR
    / "c017_canonical_tracks_v2.csv"
)

results_df.to_csv(
    TRACK_LOG_PATH,
    index=False
)

# ============================================================
# SAVE STITCH EVENTS
# ============================================================

events_df.to_csv(
    EVENT_LOG_PATH,
    index=False
)

# ============================================================
# FINAL SUMMARY
# ============================================================

elapsed = time.time() - start_time

overall_fps = (
    len(frame_paths) / elapsed
)

print("\n" + "=" * 80)
print("STITCHING V2 INSPECTION COMPLETE")
print("=" * 80)

print(
    f"\nFrames processed:       "
    f"{len(frame_paths)}"
)

print(
    f"Total tracked rows:     "
    f"{len(results_df)}"
)

print(
    f"Total stitch events:    "
    f"{len(events_df)}"
)

print(
    f"Canonical players:      "
    f"{stitcher.next_canonical_id - 1}"
)

print(
    f"Overall processing FPS: "
    f"{overall_fps:.2f}"
)

print("\nTrackStitcher statistics:")

print(
    stitcher.statistics()
)

print("\nEvent log:")
print(EVENT_LOG_PATH)

print("\nFull canonical tracking log:")
print(TRACK_LOG_PATH)

# ============================================================
# STITCH EVENTS
# ============================================================

if len(events_df) > 0:

    print("\n" + "=" * 80)
    print("FIRST STITCHING EVENTS")
    print("=" * 80)

    display(
        events_df.head(30)
    )

    # --------------------------------------------------------
    # Confidence statistics
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("STITCH CONFIDENCE STATISTICS")
    print("=" * 80)

    print(
        events_df[
            "identity_confidence"
        ].describe()
    )

    # --------------------------------------------------------
    # Speed statistics
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("STITCH SPEED STATISTICS")
    print("=" * 80)

    print(
        events_df[
            "speed_pixels_per_frame"
        ].describe()
    )

    # --------------------------------------------------------
    # Stitches per canonical player
    # --------------------------------------------------------

    stitch_summary = (
        events_df
        .groupby("canonical_id")
        .agg(
            stitch_count=(
                "canonical_id",
                "size"
            ),
            first_stitch_frame=(
                "frame_id",
                "min"
            ),
            last_stitch_frame=(
                "frame_id",
                "max"
            ),
            min_confidence=(
                "identity_confidence",
                "min"
            ),
            mean_confidence=(
                "identity_confidence",
                "mean"
            ),
            max_speed=(
                "speed_pixels_per_frame",
                "max"
            ),
            mixsort_ids=(
                "mixsort_new_id",
                lambda x: list(x)
            ),
        )
        .sort_values(
            "stitch_count",
            ascending=False
        )
    )

    print("\n" + "=" * 80)
    print("STITCHES BY CANONICAL PLAYER")
    print("=" * 80)

    display(stitch_summary)

else:

    print("\nNo stitching events detected.")

# ============================================================
# CLEANUP
# ============================================================

del tracker
del detector

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("\nGPU cleanup complete.")

REAL c017 TRACK STITCHING INSPECTION — V2

Sequence:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017

Detector:
  confidence = 0.4
  NMS IoU    = 0.4

TrackStitcher V2:
  max_distance       = 100.0
  max_frame_gap      = 30
  velocity_weight    = 0.3
  min_stitch_conf.   = 0.7

Frames found: 450

Loading player detector...
Detector loaded.

Initializing MixSort...
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MixSort ready.
TrackStitcher V2 ready.
Frame   50/450 | MixSort tracks: 19 | Canonical active: 19 | Lost:  2 | Stitches:   0 | FPS: 3.82
Frame  100/450 | MixSort tracks: 16 | Canonical active: 16 | Lost:  5 | Stitches:   0 | FPS: 4.29
Frame  150/450 | MixSort tracks: 15 | Canonical active: 15 | Lost:  6 | Stitches:   0 | FPS: 4.50
Frame  200/450 | MixSort tracks: 16 | Canonical active: 16 | Lost:  8 | Stitches:   0 |

,frame_id,mixsort_new_id,canonical_id,identity_confidence,center_x,center_y,velocity_x,velocity_y,speed_pixels_per_frame,track_history
0,351,232,22,0.771495,753.645756,342.835077,33.539763,-21.317832,39.741234,"[223, 232]"
1,377,255,24,0.885360,1035.047240,338.586431,35.976293,-0.321909,35.977733,"[226, 255]"
2,391,353,8,0.772858,1168.049961,254.168928,176.768228,8.110499,176.954194,"[201, 353]"



STITCH CONFIDENCE STATISTICS
count    3.000000
mean     0.809904
std      0.065350
min      0.771495
25%      0.772177
50%      0.772858
75%      0.829109
max      0.885360
Name: identity_confidence, dtype: float64

STITCH SPEED STATISTICS
count      3.000000
mean      84.224387
std       80.328412
min       35.977733
25%       37.859484
50%       39.741234
75%      108.347714
max      176.954194
Name: speed_pixels_per_frame, dtype: float64

STITCHES BY CANONICAL PLAYER


,stitch_count,first_stitch_frame,last_stitch_frame,min_confidence,mean_confidence,max_speed,mixsort_ids
canonical_id,,,,,,,
8,1,391,391,0.772858,0.772858,176.954194,[353]
22,1,351,351,0.771495,0.771495,39.741234,[232]
24,1,377,377,0.885360,0.885360,35.977733,[255]



GPU cleanup complete.


In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACK_LOG_PATH = (
    ROOT
    / "outputs"
    / "tracking"
    / "c017_canonical_tracks_v2.csv"
)

df = pd.read_csv(TRACK_LOG_PATH)

print("=" * 80)
print("CANONICAL ID FRAGMENTATION ANALYSIS")
print("=" * 80)

print(f"\nRows: {len(df):,}")
print(f"Frames: {df['frame_id'].nunique():,}")
print(f"MixSort IDs: {df['mixsort_id'].nunique():,}")
print(f"Canonical IDs: {df['canonical_id'].nunique():,}")

# ============================================================
# 1. CANONICAL ID LIFETIME
# ============================================================

canonical_stats = (
    df.groupby("canonical_id")
    .agg(
        first_frame=("frame_id", "min"),
        last_frame=("frame_id", "max"),
        frames_seen=("frame_id", "nunique"),
        observations=("frame_id", "size"),
        mixsort_ids=("mixsort_id", "nunique"),
    )
    .reset_index()
)

canonical_stats["lifetime"] = (
    canonical_stats["last_frame"]
    - canonical_stats["first_frame"]
    + 1
)

canonical_stats = canonical_stats.sort_values(
    "frames_seen",
    ascending=False
)

print("\n" + "=" * 80)
print("CANONICAL ID STATISTICS")
print("=" * 80)

display(canonical_stats)

# ============================================================
# 2. SHORT CANONICAL TRACKS
# ============================================================

short_tracks = canonical_stats[
    canonical_stats["frames_seen"] < 30
].sort_values(
    "frames_seen"
)

print("\n" + "=" * 80)
print("SHORT CANONICAL TRACKS (<30 observed frames)")
print("=" * 80)

print(
    f"Count: {len(short_tracks)}"
)

display(short_tracks)

# ============================================================
# 3. CANONICAL IDs WITH MULTIPLE MIXSORT IDs
# ============================================================

multi_mixsort = canonical_stats[
    canonical_stats["mixsort_ids"] > 1
].sort_values(
    "mixsort_ids",
    ascending=False
)

print("\n" + "=" * 80)
print("CANONICAL IDs ALREADY STITCHED")
print("=" * 80)

display(multi_mixsort)

# ============================================================
# 4. MIXSORT ID → CANONICAL ID MAPPING
# ============================================================

mapping = (
    df.groupby("mixsort_id")
    .agg(
        canonical_ids=(
            "canonical_id",
            lambda x: sorted(x.unique())
        ),
        first_frame=("frame_id", "min"),
        last_frame=("frame_id", "max"),
        frames=("frame_id", "nunique"),
    )
    .reset_index()
)

# Any MixSort ID associated with >1 canonical identity
ambiguous_mixsort = mapping[
    mapping["canonical_ids"].apply(len) > 1
]

print("\n" + "=" * 80)
print("AMBIGUOUS MIXSORT IDs")
print("=" * 80)

print(
    f"Count: {len(ambiguous_mixsort)}"
)

display(ambiguous_mixsort)

# ============================================================
# 5. CANONICAL ID TIMELINE
# ============================================================

print("\n" + "=" * 80)
print("CANONICAL ID TIMELINE")
print("=" * 80)

timeline = (
    canonical_stats[
        [
            "canonical_id",
            "first_frame",
            "last_frame",
            "frames_seen",
            "mixsort_ids",
        ]
    ]
    .sort_values("first_frame")
)

display(timeline)

# ============================================================
# 6. SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    f"Canonical IDs created: {len(canonical_stats)}"
)

print(
    f"Tracks with <30 frames: {len(short_tracks)}"
)

print(
    f"Canonical IDs using >1 MixSort ID: "
    f"{len(multi_mixsort)}"
)

print(
    f"MixSort IDs mapped to >1 canonical ID: "
    f"{len(ambiguous_mixsort)}"
)

print(
    "\nNext step: inspect whether the extra canonical IDs "
    "are short fragments or genuine players."
)

CANONICAL ID FRAGMENTATION ANALYSIS

Rows: 6,632
Frames: 450
MixSort IDs: 42
Canonical IDs: 39

CANONICAL ID STATISTICS


,canonical_id,first_frame,last_frame,frames_seen,observations,mixsort_ids,lifetime
16,17,1,450,431,431,1,450
0,1,1,431,429,429,1,431
7,8,1,401,388,388,2,401
11,12,1,417,386,386,1,417
12,13,1,391,381,381,1,391
13,14,1,372,364,364,1,372
18,19,1,382,360,360,1,382
20,21,3,389,358,358,1,387
3,4,1,358,357,357,1,358
19,20,1,379,340,340,1,379



SHORT CANONICAL TRACKS (<30 observed frames)
Count: 4


,canonical_id,first_frame,last_frame,frames_seen,observations,mixsort_ids,lifetime
17,18,1,1,1,1,1,1
29,30,385,386,2,2,1,2
14,15,1,2,2,2,1,2
28,29,371,372,2,2,1,2



CANONICAL IDs ALREADY STITCHED


...


AMBIGUOUS MIXSORT IDs
Count: 0


,mixsort_id,canonical_ids,first_frame,last_frame,frames



CANONICAL ID TIMELINE


,canonical_id,first_frame,last_frame,frames_seen,mixsort_ids
16,17,1,450,431,1
0,1,1,431,429,1
7,8,1,401,388,2
11,12,1,417,386,1
12,13,1,391,381,1
13,14,1,372,364,1
18,19,1,382,360,1
3,4,1,358,357,1
15,16,1,226,210,1
19,20,1,379,340,1



SUMMARY
Canonical IDs created: 39
Tracks with <30 frames: 4
Canonical IDs using >1 MixSort ID: 3
MixSort IDs mapped to >1 canonical ID: 0

Next step: inspect whether the extra canonical IDs are short fragments or genuine players.


In [20]:
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

TRACK_STATE_PATH = TRACKING_DIR / "track_state.py"

TRACK_STATE_CODE = r'''
from dataclasses import dataclass, field
from typing import Optional, Tuple, List
import math


@dataclass
class TrackState:
    """
    Persistent identity state for one canonical player.

    Velocity is estimated from a short trajectory window rather
    than a single frame-to-frame displacement.
    """

    canonical_id: int
    current_track_id: int

    position: Tuple[float, float]
    previous_position: Optional[Tuple[float, float]] = None

    velocity: Tuple[float, float] = (0.0, 0.0)

    bbox: Optional[List[float]] = None

    last_frame: int = 0
    missed_frames: int = 0

    team_id: Optional[int] = None
    identity_confidence: float = 1.0

    track_history: List[int] = field(default_factory=list)

    # Recent (frame, x, y) observations.
    position_history: List[Tuple[int, float, float]] = field(
        default_factory=list
    )

    # Number of observations used for velocity estimation.
    velocity_window: int = 5

    def update(
        self,
        position: Tuple[float, float],
        bbox: Optional[List[float]],
        frame_id: int,
        track_id: int,
        identity_confidence: Optional[float] = None,
    ):
        """
        Update the canonical track with a new observation.
        """

        self.previous_position = self.position
        self.position = position

        self.bbox = bbox
        self.current_track_id = track_id
        self.last_frame = frame_id
        self.missed_frames = 0

        if identity_confidence is not None:
            self.identity_confidence = identity_confidence

        # ----------------------------------------------------
        # Track ID history
        # ----------------------------------------------------

        if (
            not self.track_history
            or self.track_history[-1] != track_id
        ):
            self.track_history.append(track_id)

        # ----------------------------------------------------
        # Position history
        # ----------------------------------------------------

        self.position_history.append(
            (
                frame_id,
                float(position[0]),
                float(position[1]),
            )
        )

        # Keep only recent observations.
        self.position_history = self.position_history[
            -self.velocity_window:
        ]

        # ----------------------------------------------------
        # Robust velocity estimation
        # ----------------------------------------------------

        self.velocity = self._estimate_velocity()

    def _estimate_velocity(self):
        """
        Estimate velocity using linear regression over the
        recent trajectory window.

        This is much more stable than:

            current_position - previous_position

        because a single noisy detection can otherwise produce
        an unrealistically large velocity.
        """

        if len(self.position_history) < 2:
            return (0.0, 0.0)

        frames = [
            float(item[0])
            for item in self.position_history
        ]

        xs = [
            float(item[1])
            for item in self.position_history
        ]

        ys = [
            float(item[2])
            for item in self.position_history
        ]

        # ----------------------------------------------------
        # Mean-center frame values
        # ----------------------------------------------------

        mean_frame = sum(frames) / len(frames)

        centered_frames = [
            f - mean_frame
            for f in frames
        ]

        denominator = sum(
            value * value
            for value in centered_frames
        )

        if denominator <= 1e-12:
            return (0.0, 0.0)

        mean_x = sum(xs) / len(xs)
        mean_y = sum(ys) / len(ys)

        velocity_x = sum(
            (f - mean_frame) * (x - mean_x)
            for f, x in zip(frames, xs)
        ) / denominator

        velocity_y = sum(
            (f - mean_frame) * (y - mean_y)
            for f, y in zip(frames, ys)
        ) / denominator

        return (
            float(velocity_x),
            float(velocity_y),
        )

    def mark_missing(self):
        """
        Mark this canonical identity as temporarily missing.
        """
        self.missed_frames += 1

    def speed(self):
        """
        Current speed in pixels/frame.
        """
        return math.sqrt(
            self.velocity[0] ** 2
            + self.velocity[1] ** 2
        )
'''

TRACKING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRACK_STATE_PATH.write_text(
    TRACK_STATE_CODE,
    encoding="utf-8"
)

print("=" * 70)
print("track_state.py UPDATED")
print("=" * 70)

print(f"\nPath:")
print(TRACK_STATE_PATH)

print("\nVelocity estimation:")
print("  Method: linear regression")
print("  Window: 5 recent observations")
print("  Unit: pixels/frame")

# Verify import using the fresh file
import sys

for module_name in ["track_state"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

from track_state import TrackState

# ------------------------------------------------------------
# Small sanity test
# ------------------------------------------------------------

state = TrackState(
    canonical_id=1,
    current_track_id=101,
    position=(500.0, 300.0),
)

test_positions = [
    (1, (500.0, 300.0)),
    (2, (505.0, 302.0)),
    (3, (510.0, 304.0)),
    (4, (515.0, 306.0)),
    (5, (520.0, 308.0)),
]

for frame_id, position in test_positions:
    state.update(
        position=position,
        bbox=[480, 280, 520, 360],
        frame_id=frame_id,
        track_id=101,
    )

print("\nSanity test:")
print(f"Velocity: {state.velocity}")
print(f"Speed:    {state.speed():.4f} px/frame")

print("\nExpected approximately:")
print("Velocity: (5.0, 2.0)")
print("Speed:    5.3852 px/frame")

print("\n✓ TrackState velocity fix is ready.")

track_state.py UPDATED

Path:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/track_state.py

Velocity estimation:
  Method: linear regression
  Window: 5 recent observations
  Unit: pixels/frame

Sanity test:
Velocity: (5.0, 2.0)
Speed:    5.3852 px/frame

Expected approximately:
Velocity: (5.0, 2.0)
Speed:    5.3852 px/frame

✓ TrackState velocity fix is ready.


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

path = (
    ROOT
    / "outputs"
    / "tracking"
    / "c017_canonical_tracks_v2.csv"
)

df = pd.read_csv(path)

print("=" * 80)
print("ACTUAL VELOCITY RANGE — c017")
print("=" * 80)

# Calculate frame-to-frame displacement for diagnostic purposes
df = df.sort_values(
    ["canonical_id", "frame_id"]
).copy()

df["dx"] = (
    df.groupby("canonical_id")["center_x"]
    .diff()
)

df["dy"] = (
    df.groupby("canonical_id")["center_y"]
    .diff()
)

df["raw_speed"] = np.sqrt(
    df["dx"] ** 2 +
    df["dy"] ** 2
)

print("\nRaw frame-to-frame speed:")
print(
    df["raw_speed"]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

print("\nLargest frame-to-frame jumps:")

largest = (
    df[
        [
            "frame_id",
            "canonical_id",
            "mixsort_id",
            "center_x",
            "center_y",
            "raw_speed",
        ]
    ]
    .dropna()
    .sort_values(
        "raw_speed",
        ascending=False
    )
    .head(20)
)

display(largest)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

ACTUAL VELOCITY RANGE — c017

Raw frame-to-frame speed:
count    6593.000000
mean        4.864876
std         7.039693
min         0.042659
50%         3.590180
90%         8.800566
95%        12.089687
99%        26.487551
99.9%      79.737302
max       244.008208
Name: raw_speed, dtype: float64

Largest frame-to-frame jumps:


,frame_id,canonical_id,mixsort_id,center_x,center_y,raw_speed
5869,385,24,255,1252.063819,344.188524,244.008208
5865,384,24,226,1027.508942,248.715426,218.734060
5915,391,8,353,1168.049961,254.168928,176.954194
5838,381,24,255,1133.344339,343.134498,127.717616
5834,380,24,226,1011.639862,304.407118,100.815436
5873,385,12,205,925.808308,276.089984,91.291040
5902,389,21,214,66.847770,387.108501,84.014585
3142,191,22,223,793.660320,293.258042,76.789445
5840,381,13,206,1011.610947,247.253766,68.600896
5866,384,22,232,1127.608168,300.281680,63.521737



DONE


In [23]:
import sys
from pathlib import Path

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

# Reload both modules
for name in [
    "track_state",
    "track_stitcher",
]:
    if name in sys.modules:
        del sys.modules[name]

from track_stitcher import TrackStitcher

stitcher = TrackStitcher(
    max_distance=100,
    max_frame_gap=30,
    velocity_weight=0.30,
    min_stitch_confidence=0.70,
    max_prediction_speed=30.0,
)

# ------------------------------------------------------------
# Frame 1
# ------------------------------------------------------------

r1 = stitcher.update(
    [
        {
            "track_id": 101,
            "bbox": [480, 280, 520, 360],
            "score": 0.95,
        }
    ],
    frame_id=1,
)

# ------------------------------------------------------------
# Frame 2
# ------------------------------------------------------------

r2 = stitcher.update(
    [
        {
            "track_id": 101,
            "bbox": [485, 282, 525, 362],
            "score": 0.95,
        }
    ],
    frame_id=2,
)

# ------------------------------------------------------------
# Frame 3
# MixSort ID changes
# ------------------------------------------------------------

r3 = stitcher.update(
    [
        {
            "track_id": 202,
            "bbox": [490, 284, 530, 364],
            "score": 0.94,
        }
    ],
    frame_id=3,
)

print("=" * 70)
print("TRACK STITCHER V3 TEST")
print("=" * 70)

print("\nFrame 1:")
print(r1)

print("\nFrame 2:")
print(r2)

print("\nFrame 3:")
print(r3)

print("\nStatistics:")
print(stitcher.statistics())

print("\nState velocity:")

state = stitcher.states[1]

print(
    "Velocity:",
    state.velocity
)

print(
    "Speed:",
    state.speed()
)

print(
    "Position history:",
    state.position_history
)

assert r1[0]["canonical_id"] == 1
assert r2[0]["canonical_id"] == 1
assert r3[0]["canonical_id"] == 1
assert r3[0]["stitched"] is True
assert stitcher.total_stitches == 1

print("\n✓ V3 controlled test PASSED")

TRACK STITCHER V3 TEST

Frame 1:
[{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [480.0, 280.0, 520.0, 360.0], 'center': (500.0, 320.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]

Frame 2:
[{'mixsort_id': 101, 'canonical_id': 1, 'bbox': [485.0, 282.0, 525.0, 362.0], 'center': (505.0, 322.0), 'score': 0.95, 'stitched': False, 'identity_confidence': 1.0}]

Frame 3:
[{'mixsort_id': 202, 'canonical_id': 1, 'bbox': [490.0, 284.0, 530.0, 364.0], 'center': (510.0, 324.0), 'score': 0.94, 'stitched': True, 'identity_confidence': 0.9999999999999999}]

Statistics:
{'active_players': 1, 'lost_players': 0, 'canonical_players_created': 1, 'total_stitches': 1, 'total_new_players': 1}

State velocity:
Velocity: (5.0, 2.0)
Speed: 5.385164807134504
Position history: [(1, 500.0, 320.0), (2, 505.0, 322.0), (3, 510.0, 324.0)]

✓ V3 controlled test PASSED


In [24]:
from pathlib import Path
import sys
import gc
import time

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
MIXSORT_DIR = ROOT / "ai-engine" / "third_party" / "MixSort"

SEQ_DIR = (
    ROOT
    / "datasets"
    / "football"
    / "benchmarks"
    / "sportsmot"
    / "extracted"
    / "val"
    / "v_2QhNRucNC7E_c017"
)

MODEL_PATH = (
    ROOT
    / "ai-engine"
    / "models"
    / "football_player_detector.pt"
)

OUTPUT_DIR = ROOT / "outputs" / "tracking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACK_LOG_PATH = (
    OUTPUT_DIR / "c017_canonical_tracks_v3.csv"
)

EVENT_LOG_PATH = (
    OUTPUT_DIR / "c017_track_stitch_events_v3.csv"
)

# ============================================================
# CONFIGURATION
# ============================================================

DETECTOR_CONF = 0.40
DETECTOR_IOU = 0.40
FPS = 25

# MixSort final configuration
TRACK_THRESH = 0.60
TRACK_BUFFER = 30
MATCH_THRESH = 0.90
MIX_IOU_THRESH = 0.30
ALPHA = 0.80

# TrackStitcher V3
STITCH_MAX_DISTANCE = 100.0
STITCH_MAX_FRAME_GAP = 30
STITCH_VELOCITY_WEIGHT = 0.30
MIN_STITCH_CONFIDENCE = 0.70
MAX_PREDICTION_SPEED = 30.0

# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("REAL c017 TRACK STITCHING — V3 / 450 FRAME TEST")
print("=" * 80)

print("\nSequence:")
print(SEQ_DIR)

print("\nDetector:")
print(f"  confidence = {DETECTOR_CONF}")
print(f"  NMS IoU    = {DETECTOR_IOU}")

print("\nMixSort:")
print(f"  track_thresh = {TRACK_THRESH}")
print(f"  track_buffer = {TRACK_BUFFER}")
print(f"  match_thresh = {MATCH_THRESH}")
print(f"  iou_thresh   = {MIX_IOU_THRESH}")
print(f"  alpha        = {ALPHA}")

print("\nTrackStitcher V3:")
print(f"  max_distance          = {STITCH_MAX_DISTANCE}")
print(f"  max_frame_gap         = {STITCH_MAX_FRAME_GAP}")
print(f"  velocity_weight       = {STITCH_VELOCITY_WEIGHT}")
print(f"  min_stitch_confidence = {MIN_STITCH_CONFIDENCE}")
print(f"  max_prediction_speed  = {MAX_PREDICTION_SPEED}")

# ============================================================
# FIND FRAMES
# ============================================================

IMG_DIR = SEQ_DIR / "img1"

frame_paths = sorted(
    IMG_DIR.glob("*.jpg")
)

if not frame_paths:
    raise FileNotFoundError(
        f"No frames found: {IMG_DIR}"
    )

print(f"\nFrames found: {len(frame_paths)}")

# ============================================================
# IMPORT LATEST TRACKING MODULES
# ============================================================

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

# Force fresh imports
for module_name in [
    "track_state",
    "track_stitcher",
    "mixsort_tracker",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from track_stitcher import TrackStitcher
from mixsort_tracker import AthletiQMixSort

# ============================================================
# LOAD DETECTOR
# ============================================================

print("\nLoading player detector...")

detector = YOLO(
    str(MODEL_PATH)
)

print("Detector loaded.")

# ============================================================
# INITIALIZE MIXSORT
# ============================================================

print("\nInitializing MixSort...")

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=FPS,
)

print("MixSort ready.")

# ============================================================
# INITIALIZE TRACK STITCHER V3
# ============================================================

print("\nInitializing TrackStitcher V3...")

stitcher = TrackStitcher(
    max_distance=STITCH_MAX_DISTANCE,
    max_frame_gap=STITCH_MAX_FRAME_GAP,
    velocity_weight=STITCH_VELOCITY_WEIGHT,
    min_stitch_confidence=MIN_STITCH_CONFIDENCE,
    max_prediction_speed=MAX_PREDICTION_SPEED,
)

print("TrackStitcher V3 ready.")

# ============================================================
# STORAGE
# ============================================================

all_results = []
stitch_events = []

start_time = time.time()

# ============================================================
# PROCESS 450 FRAMES
# ============================================================

for frame_index, frame_path in enumerate(
    frame_paths,
    start=1,
):

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        print(
            f"WARNING: failed to read {frame_path}"
        )
        continue

    # --------------------------------------------------------
    # PLAYER DETECTION
    # --------------------------------------------------------

    result = detector.predict(
        frame,
        conf=DETECTOR_CONF,
        iou=DETECTOR_IOU,
        device=0,
        verbose=False,
    )[0]

    if (
        result.boxes is not None
        and len(result.boxes) > 0
    ):

        boxes = (
            result.boxes
            .xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result.boxes
            .conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [boxes, scores]
        ).astype(
            np.float32
        )

    else:

        detections = np.empty(
            (0, 5),
            dtype=np.float32
        )

    # --------------------------------------------------------
    # MIXSORT
    # --------------------------------------------------------

    mixsort_tracks = tracker.update(
        detections.copy(),
        frame,
    )

    # --------------------------------------------------------
    # TRACK STITCHER V3
    # --------------------------------------------------------

    canonical_tracks = stitcher.update(
        mixsort_tracks,
        frame_id=frame_index,
    )

    # --------------------------------------------------------
    # SAVE TRACK RESULTS
    # --------------------------------------------------------

    for track in canonical_tracks:

        all_results.append({
            "frame_id": frame_index,

            "mixsort_id": track[
                "mixsort_id"
            ],

            "canonical_id": track[
                "canonical_id"
            ],

            "x1": track["bbox"][0],
            "y1": track["bbox"][1],
            "x2": track["bbox"][2],
            "y2": track["bbox"][3],

            "center_x": track[
                "center"
            ][0],

            "center_y": track[
                "center"
            ][1],

            "score": track["score"],

            "stitched": track[
                "stitched"
            ],

            "identity_confidence": track[
                "identity_confidence"
            ],
        })

    # --------------------------------------------------------
    # RECORD STITCH EVENTS
    # --------------------------------------------------------

    for track in canonical_tracks:

        if not track["stitched"]:
            continue

        canonical_id = track[
            "canonical_id"
        ]

        state = stitcher.states[
            canonical_id
        ]

        stitch_events.append({
            "frame_id": frame_index,

            "mixsort_new_id": track[
                "mixsort_id"
            ],

            "canonical_id": canonical_id,

            "identity_confidence": track[
                "identity_confidence"
            ],

            "center_x": track[
                "center"
            ][0],

            "center_y": track[
                "center"
            ][1],

            "velocity_x": state.velocity[0],

            "velocity_y": state.velocity[1],

            "speed_pixels_per_frame": (
                state.speed()
            ),

            "track_history": str(
                state.track_history
            ),
        })

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if frame_index % 50 == 0:

        elapsed = (
            time.time()
            - start_time
        )

        current_fps = (
            frame_index
            / elapsed
        )

        print(
            f"Frame {frame_index:4d}/"
            f"{len(frame_paths)} | "
            f"MixSort: {len(mixsort_tracks):2d} | "
            f"Canonical active: "
            f"{len(canonical_tracks):2d} | "
            f"Lost: {len(stitcher.lost_ids):2d} | "
            f"Stitches: "
            f"{len(stitch_events):3d} | "
            f"FPS: {current_fps:.2f}"
        )

# ============================================================
# DATAFRAMES
# ============================================================

results_df = pd.DataFrame(
    all_results
)

events_df = pd.DataFrame(
    stitch_events
)

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    TRACK_LOG_PATH,
    index=False,
)

events_df.to_csv(
    EVENT_LOG_PATH,
    index=False,
)

# ============================================================
# FINAL PERFORMANCE
# ============================================================

elapsed = (
    time.time()
    - start_time
)

overall_fps = (
    len(frame_paths)
    / elapsed
)

print("\n" + "=" * 80)
print("V3 450-FRAME TEST COMPLETE")
print("=" * 80)

print(
    f"\nFrames processed:       "
    f"{len(frame_paths)}"
)

print(
    f"Total tracked rows:     "
    f"{len(results_df):,}"
)

print(
    f"MixSort IDs:            "
    f"{results_df['mixsort_id'].nunique()}"
)

print(
    f"Canonical players:      "
    f"{results_df['canonical_id'].nunique()}"
)

print(
    f"Total stitch events:    "
    f"{len(events_df)}"
)

print(
    f"Processing FPS:         "
    f"{overall_fps:.2f}"
)

# ============================================================
# TRACKER STATISTICS
# ============================================================

print("\nTrackStitcher statistics:")

print(
    stitcher.statistics()
)

# ============================================================
# OUTPUT PATHS
# ============================================================

print("\nSaved files:")

print(
    f"Track log:\n{TRACK_LOG_PATH}"
)

print(
    f"\nEvent log:\n{EVENT_LOG_PATH}"
)

# ============================================================
# STITCH EVENTS
# ============================================================

if len(events_df) > 0:

    print("\n" + "=" * 80)
    print("STITCH EVENTS")
    print("=" * 80)

    display(
        events_df
    )

    # --------------------------------------------------------
    # Confidence statistics
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("STITCH CONFIDENCE")
    print("=" * 80)

    print(
        events_df[
            "identity_confidence"
        ].describe()
    )

    # --------------------------------------------------------
    # Velocity statistics
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("STITCH VELOCITY")
    print("=" * 80)

    print(
        events_df[
            "speed_pixels_per_frame"
        ].describe()
    )

    # --------------------------------------------------------
    # Stitch summary per canonical ID
    # --------------------------------------------------------

    stitch_summary = (
        events_df
        .groupby("canonical_id")
        .agg(
            stitch_count=(
                "canonical_id",
                "size",
            ),

            first_stitch_frame=(
                "frame_id",
                "min",
            ),

            last_stitch_frame=(
                "frame_id",
                "max",
            ),

            min_confidence=(
                "identity_confidence",
                "min",
            ),

            mean_confidence=(
                "identity_confidence",
                "mean",
            ),

            max_speed=(
                "speed_pixels_per_frame",
                "max",
            ),

            mixsort_ids=(
                "mixsort_new_id",
                lambda x: list(x),
            ),
        )
        .sort_values(
            "stitch_count",
            ascending=False,
        )
    )

    print("\n" + "=" * 80)
    print("STITCHES BY CANONICAL PLAYER")
    print("=" * 80)

    display(
        stitch_summary
    )

else:

    print(
        "\nNo stitching events detected."
    )

# ============================================================
# CANONICAL ID FRAGMENTATION
# ============================================================

canonical_stats = (
    results_df
    .groupby("canonical_id")
    .agg(
        first_frame=(
            "frame_id",
            "min",
        ),

        last_frame=(
            "frame_id",
            "max",
        ),

        frames_seen=(
            "frame_id",
            "nunique",
        ),

        observations=(
            "frame_id",
            "size",
        ),

        mixsort_ids=(
            "mixsort_id",
            "nunique",
        ),
    )
    .reset_index()
)

canonical_stats["lifetime"] = (
    canonical_stats["last_frame"]
    - canonical_stats["first_frame"]
    + 1
)

short_tracks = canonical_stats[
    canonical_stats["frames_seen"] < 30
]

multi_mixsort = canonical_stats[
    canonical_stats["mixsort_ids"] > 1
]

print("\n" + "=" * 80)
print("CANONICAL FRAGMENTATION SUMMARY")
print("=" * 80)

print(
    f"Canonical IDs:                 "
    f"{len(canonical_stats)}"
)

print(
    f"Tracks <30 frames:             "
    f"{len(short_tracks)}"
)

print(
    f"Canonical IDs with >1 MixSort: "
    f"{len(multi_mixsort)}"
)

# ============================================================
# GPU CLEANUP
# ============================================================

del tracker
del detector

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("\nGPU cleanup complete.")

REAL c017 TRACK STITCHING — V3 / 450 FRAME TEST

Sequence:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017

Detector:
  confidence = 0.4
  NMS IoU    = 0.4

MixSort:
  track_thresh = 0.6
  track_buffer = 30
  match_thresh = 0.9
  iou_thresh   = 0.3
  alpha        = 0.8

TrackStitcher V3:
  max_distance          = 100.0
  max_frame_gap         = 30
  velocity_weight       = 0.3
  min_stitch_confidence = 0.7
  max_prediction_speed  = 30.0

Frames found: 450

Loading player detector...
Detector loaded.

Initializing MixSort...
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MixSort ready.

Initializing TrackStitcher V3...
TrackStitcher V3 ready.
Frame   50/450 | MixSort: 19 | Canonical active: 19 | Lost:  0 | Stitches:   0 | FPS: 4.35
Frame  100/450 | MixSort: 16 | Canonical active: 16 | Lost:  2 | Stitches:   0 | FPS: 4.46
Fr

,frame_id,mixsort_new_id,canonical_id,identity_confidence,center_x,center_y,velocity_x,velocity_y,speed_pixels_per_frame,track_history
0,377,448,24,0.831356,1035.04724,338.586431,19.21139,1.75832,19.291687,"[419, 448]"



STITCH CONFIDENCE
count    1.000000
mean     0.831356
std           NaN
min      0.831356
25%      0.831356
50%      0.831356
75%      0.831356
max      0.831356
Name: identity_confidence, dtype: float64

STITCH VELOCITY
count     1.000000
mean     19.291687
std            NaN
min      19.291687
25%      19.291687
50%      19.291687
75%      19.291687
max      19.291687
Name: speed_pixels_per_frame, dtype: float64

STITCHES BY CANONICAL PLAYER


,stitch_count,first_stitch_frame,last_stitch_frame,min_confidence,mean_confidence,max_speed,mixsort_ids
canonical_id,,,,,,,
24,1,377,377,0.831356,0.831356,19.291687,[448]



CANONICAL FRAGMENTATION SUMMARY
Canonical IDs:                 41
Tracks <30 frames:             5
Canonical IDs with >1 MixSort: 1

GPU cleanup complete.


In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import configparser

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

SEQ_DIR = (
    ROOT
    / "datasets"
    / "football"
    / "benchmarks"
    / "sportsmot"
    / "extracted"
    / "val"
    / "v_2QhNRucNC7E_c017"
)

GT_PATH = SEQ_DIR / "gt" / "gt.txt"

TRACK_LOG_PATH = (
    ROOT
    / "outputs"
    / "tracking"
    / "c017_canonical_tracks_v3.csv"
)

print("=" * 80)
print("SPORTSMOT GT vs MIXSORT FRAGMENTATION ANALYSIS")
print("=" * 80)

print("\nGT:")
print(GT_PATH)

print("\nMixSort tracking:")
print(TRACK_LOG_PATH)

# ============================================================
# LOAD GT
# ============================================================

gt_columns = [
    "frame_id",
    "gt_id",
    "x",
    "y",
    "w",
    "h",
    "confidence",
    "class_id",
    "visibility",
    "unused",
]

gt = pd.read_csv(
    GT_PATH,
    header=None,
    names=gt_columns,
)

# SportsMOT player class
# We keep only valid player annotations.
gt = gt[
    gt["gt_id"] > 0
].copy()

print("\nGT rows:", len(gt))
print(
    "GT unique players:",
    gt["gt_id"].nunique()
)

print(
    "GT frames:",
    gt["frame_id"].nunique()
)

# ============================================================
# LOAD V3 MIXSORT TRACKS
# ============================================================

tracks = pd.read_csv(
    TRACK_LOG_PATH
)

print("\nMixSort rows:", len(tracks))

print(
    "MixSort IDs:",
    tracks["mixsort_id"].nunique()
)

print(
    "Canonical IDs:",
    tracks["canonical_id"].nunique()
)

# ============================================================
# CONVERT BBOXES TO XYWH
# ============================================================

tracks["x"] = tracks["x1"]
tracks["y"] = tracks["y1"]

tracks["w"] = (
    tracks["x2"]
    - tracks["x1"]
)

tracks["h"] = (
    tracks["y2"]
    - tracks["y1"]
)

# ============================================================
# IOU FUNCTION
# ============================================================

def bbox_iou(
    box_a,
    box_b,
):
    """
    box = [x, y, w, h]
    """

    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b

    ax2 = ax + aw
    ay2 = ay + ah

    bx2 = bx + bw
    by2 = by + bh

    ix1 = max(ax, bx)
    iy1 = max(ay, by)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0.0,
        ix2 - ix1,
    )

    ih = max(
        0.0,
        iy2 - iy1,
    )

    intersection = iw * ih

    area_a = aw * ah
    area_b = bw * bh

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


# ============================================================
# MATCH MIXSORT → GT
# ============================================================

IOU_THRESHOLD = 0.50

matches = []

# Process frame by frame
for frame_id, frame_tracks in tracks.groupby(
    "frame_id"
):

    frame_gt = gt[
        gt["frame_id"] == frame_id
    ]

    if frame_gt.empty:
        continue

    # --------------------------------------------------------
    # Compute all possible IoUs
    # --------------------------------------------------------

    for _, tr in frame_tracks.iterrows():

        track_box = [
            tr["x"],
            tr["y"],
            tr["w"],
            tr["h"],
        ]

        best_gt_id = None
        best_iou = 0.0

        for _, g in frame_gt.iterrows():

            gt_box = [
                g["x"],
                g["y"],
                g["w"],
                g["h"],
            ]

            iou = bbox_iou(
                track_box,
                gt_box,
            )

            if iou > best_iou:

                best_iou = iou
                best_gt_id = int(
                    g["gt_id"]
                )

        if (
            best_gt_id is not None
            and best_iou >= IOU_THRESHOLD
        ):

            matches.append({
                "frame_id": int(frame_id),

                "mixsort_id": int(
                    tr["mixsort_id"]
                ),

                "canonical_id": int(
                    tr["canonical_id"]
                ),

                "gt_id": best_gt_id,

                "iou": best_iou,
            })

matches_df = pd.DataFrame(
    matches
)

print("\n" + "=" * 80)
print("FRAME-LEVEL MATCHING")
print("=" * 80)

print(
    f"Matched observations: "
    f"{len(matches_df):,}"
)

print(
    f"Unique MixSort IDs matched: "
    f"{matches_df['mixsort_id'].nunique()}"
)

print(
    f"Unique GT players matched: "
    f"{matches_df['gt_id'].nunique()}"
)

print(
    f"Mean IoU: "
    f"{matches_df['iou'].mean():.4f}"
)

# ============================================================
# 1. MIXSORT ID → GT ID
# ============================================================

mixsort_to_gt = (
    matches_df
    .groupby("mixsort_id")
    .agg(
        gt_ids=(
            "gt_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        gt_id_count=(
            "gt_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

# ============================================================
# 2. GT ID → MIXSORT IDs
# ============================================================

gt_to_mixsort = (
    matches_df
    .groupby("gt_id")
    .agg(
        mixsort_ids=(
            "mixsort_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        mixsort_id_count=(
            "mixsort_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

# ============================================================
# 3. MOST FRAGMENTED GT PLAYERS
# ============================================================

most_fragmented = (
    gt_to_mixsort
    .sort_values(
        [
            "mixsort_id_count",
            "matched_frames",
        ],
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("MOST FRAGMENTED GT PLAYERS")
print("=" * 80)

display(
    most_fragmented
)

# ============================================================
# 4. MIXSORT IDS ASSOCIATED WITH MULTIPLE GT PLAYERS
# ============================================================

ambiguous_mixsort = (
    mixsort_to_gt[
        mixsort_to_gt[
            "gt_id_count"
        ] > 1
    ]
    .sort_values(
        "gt_id_count",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("MIXSORT IDs ASSOCIATED WITH MULTIPLE GT PLAYERS")
print("=" * 80)

print(
    f"Count: {len(ambiguous_mixsort)}"
)

display(
    ambiguous_mixsort
)

# ============================================================
# 5. CLEAN MIXSORT FRAGMENTS
# ============================================================

clean_fragments = (
    mixsort_to_gt[
        mixsort_to_gt[
            "gt_id_count"
        ] == 1
    ]
    .copy()
)

print("\n" + "=" * 80)
print("CLEAN MIXSORT FRAGMENTS")
print("=" * 80)

print(
    f"MixSort IDs belonging cleanly to "
    f"one GT player: {len(clean_fragments)}"
)

display(
    clean_fragments.sort_values(
        "matched_frames",
        ascending=False,
    )
)

# ============================================================
# 6. FRAGMENTATION DISTRIBUTION
# ============================================================

fragmentation_distribution = (
    gt_to_mixsort[
        "mixsort_id_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "number_of_mixsort_ids"
    )
    .reset_index(
        name="number_of_gt_players"
    )
)

print("\n" + "=" * 80)
print("GT PLAYER FRAGMENTATION DISTRIBUTION")
print("=" * 80)

display(
    fragmentation_distribution
)

# ============================================================
# 7. CANONICAL IDs VS GT
# ============================================================

canonical_to_gt = (
    matches_df
    .groupby("canonical_id")
    .agg(
        gt_ids=(
            "gt_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        gt_id_count=(
            "gt_id",
            "nunique",
        ),

        mixsort_ids=(
            "mixsort_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

ambiguous_canonical = (
    canonical_to_gt[
        canonical_to_gt[
            "gt_id_count"
        ] > 1
    ]
)

print("\n" + "=" * 80)
print("CANONICAL IDs ASSOCIATED WITH MULTIPLE GT PLAYERS")
print("=" * 80)

print(
    f"Count: {len(ambiguous_canonical)}"
)

display(
    ambiguous_canonical
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL FRAGMENTATION SUMMARY")
print("=" * 80)

print(
    f"GT players:                    "
    f"{gt['gt_id'].nunique()}"
)

print(
    f"MixSort IDs:                   "
    f"{tracks['mixsort_id'].nunique()}"
)

print(
    f"Canonical IDs:                 "
    f"{tracks['canonical_id'].nunique()}"
)

print(
    f"GT players with >1 MixSort ID: "
    f"{len(gt_to_mixsort[gt_to_mixsort['mixsort_id_count'] > 1])}"
)

print(
    f"MixSort IDs → multiple GT:     "
    f"{len(ambiguous_mixsort)}"
)

print(
    f"Canonical IDs → multiple GT:   "
    f"{len(ambiguous_canonical)}"
)

print(
    f"Mean matched IoU:              "
    f"{matches_df['iou'].mean():.4f}"
)

print("\nDone.")

SPORTSMOT GT vs MIXSORT FRAGMENTATION ANALYSIS

GT:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017/gt/gt.txt

MixSort tracking:
/mnt/d/My Projects/AthletiQ/AthletiQ/outputs/tracking/c017_canonical_tracks_v3.csv

GT rows: 6183
GT unique players: 20
GT frames: 450

MixSort rows: 6632
MixSort IDs: 42
Canonical IDs: 41

FRAME-LEVEL MATCHING
Matched observations: 5,760
Unique MixSort IDs matched: 37
Unique GT players matched: 20
Mean IoU: 0.8449

MOST FRAGMENTED GT PLAYERS


,gt_id,mixsort_ids,mixsort_id_count,matched_frames,mean_iou
14,15,"[388, 405, 417, 419, 448]",5,315,0.830889
1,2,"[393, 416, 417, 419, 425]",5,311,0.824267
3,4,"[400, 403, 545, 574]",4,426,0.842647
17,18,"[403, 406, 417, 574]",4,422,0.815181
4,5,"[398, 405, 406, 417]",4,373,0.825423
6,7,"[394, 398, 419, 546]",4,368,0.843189
2,3,"[395, 403, 405, 407]",4,229,0.843876
13,14,"[405, 407, 556]",3,413,0.835441
11,12,"[397, 571, 575]",3,291,0.865746
5,6,"[396, 407, 576]",3,214,0.879447



MIXSORT IDs ASSOCIATED WITH MULTIPLE GT PLAYERS
Count: 10


,mixsort_id,gt_ids,gt_id_count,matched_frames,mean_iou
20,417,"[2, 5, 15, 18]",4,204,0.840010
16,405,"[3, 5, 14, 15]",4,10,0.593811
21,419,"[1, 2, 7, 15]",4,183,0.854839
6,394,"[7, 13, 17]",3,375,0.854671
18,407,"[3, 6, 14]",3,358,0.835765
15,403,"[3, 4, 18]",3,430,0.823029
10,398,"[5, 7]",2,386,0.836406
17,406,"[5, 18]",2,340,0.838475
33,571,"[8, 12]",2,45,0.837967
34,574,"[4, 18]",2,50,0.787263



CLEAN MIXSORT FRAGMENTS
MixSort IDs belonging cleanly to one GT player: 27


...


GT PLAYER FRAGMENTATION DISTRIBUTION


,number_of_mixsort_ids,number_of_gt_players
0,1,4
1,2,5
2,3,4
3,4,5
4,5,2



CANONICAL IDs ASSOCIATED WITH MULTIPLE GT PLAYERS
Count: 10


,canonical_id,gt_ids,gt_id_count,mixsort_ids,matched_frames,mean_iou
6,8,"[7, 13, 17]",3,1,375,0.854671
10,12,"[5, 7]",2,1,386,0.836406
15,17,"[3, 4, 18]",3,1,430,0.823029
16,19,"[3, 5, 14, 15]",4,1,10,0.593811
17,20,"[5, 18]",2,1,340,0.838475
18,21,"[3, 6, 14]",3,1,358,0.835765
20,23,"[2, 5, 15, 18]",4,1,204,0.840010
21,24,"[1, 2, 7, 15]",4,2,191,0.849366
32,38,"[8, 12]",2,1,45,0.837967
33,39,"[4, 18]",2,1,50,0.787263



FINAL FRAGMENTATION SUMMARY
GT players:                    20
MixSort IDs:                   42
Canonical IDs:                 41
GT players with >1 MixSort ID: 16
MixSort IDs → multiple GT:     10
Canonical IDs → multiple GT:   10
Mean matched IoU:              0.8449

Done.


In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.optimize import linear_sum_assignment

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

SEQ_DIR = (
    ROOT
    / "datasets"
    / "football"
    / "benchmarks"
    / "sportsmot"
    / "extracted"
    / "val"
    / "v_2QhNRucNC7E_c017"
)

GT_PATH = SEQ_DIR / "gt" / "gt.txt"

TRACK_PATH = (
    ROOT
    / "outputs"
    / "tracking"
    / "c017_canonical_tracks_v3.csv"
)

OUTPUT_DIR = ROOT / "outputs" / "tracking"

MATCH_LOG_PATH = (
    OUTPUT_DIR
    / "c017_hungarian_gt_matches_v1.csv"
)

SWITCH_LOG_PATH = (
    OUTPUT_DIR
    / "c017_mixsort_switch_events_v1.csv"
)

# ============================================================
# CONFIG
# ============================================================

IOU_THRESHOLD = 0.50

# ============================================================
# LOAD GT
# ============================================================

gt_columns = [
    "frame_id",
    "gt_id",
    "x",
    "y",
    "w",
    "h",
    "confidence",
    "class_id",
    "visibility",
    "unused",
]

gt = pd.read_csv(
    GT_PATH,
    header=None,
    names=gt_columns,
)

gt = gt[
    gt["gt_id"] > 0
].copy()

# ============================================================
# LOAD TRACKS
# ============================================================

tracks = pd.read_csv(
    TRACK_PATH
)

tracks["x"] = tracks["x1"]
tracks["y"] = tracks["y1"]

tracks["w"] = (
    tracks["x2"]
    - tracks["x1"]
)

tracks["h"] = (
    tracks["y2"]
    - tracks["y1"]
)

print("=" * 80)
print("ONE-TO-ONE HUNGARIAN GT IDENTITY ANALYSIS")
print("=" * 80)

print(
    f"\nGT rows: {len(gt):,}"
)

print(
    f"GT players: {gt['gt_id'].nunique()}"
)

print(
    f"Track rows: {len(tracks):,}"
)

print(
    f"MixSort IDs: {tracks['mixsort_id'].nunique()}"
)

# ============================================================
# IOU FUNCTION
# ============================================================

def bbox_iou_matrix(
    track_boxes,
    gt_boxes,
):
    """
    Compute IoU matrix.

    track_boxes: N x 4
    gt_boxes:    M x 4

    Format:
        [x, y, w, h]
    """

    if len(track_boxes) == 0:
        return np.empty(
            (0, len(gt_boxes))
        )

    if len(gt_boxes) == 0:
        return np.empty(
            (len(track_boxes), 0)
        )

    track_boxes = np.asarray(
        track_boxes,
        dtype=np.float64,
    )

    gt_boxes = np.asarray(
        gt_boxes,
        dtype=np.float64,
    )

    tx1 = track_boxes[:, 0][:, None]
    ty1 = track_boxes[:, 1][:, None]

    tx2 = (
        track_boxes[:, 0]
        + track_boxes[:, 2]
    )[:, None]

    ty2 = (
        track_boxes[:, 1]
        + track_boxes[:, 3]
    )[:, None]

    gx1 = gt_boxes[:, 0][None, :]
    gy1 = gt_boxes[:, 1][None, :]

    gx2 = (
        gt_boxes[:, 0]
        + gt_boxes[:, 2]
    )[None, :]

    gy2 = (
        gt_boxes[:, 1]
        + gt_boxes[:, 3]
    )[None, :]

    ix1 = np.maximum(
        tx1,
        gx1,
    )

    iy1 = np.maximum(
        ty1,
        gy1,
    )

    ix2 = np.minimum(
        tx2,
        gx2,
    )

    iy2 = np.minimum(
        ty2,
        gy2,
    )

    iw = np.maximum(
        0,
        ix2 - ix1,
    )

    ih = np.maximum(
        0,
        iy2 - iy1,
    )

    intersection = (
        iw * ih
    )

    track_area = (
        track_boxes[:, 2]
        * track_boxes[:, 3]
    )[:, None]

    gt_area = (
        gt_boxes[:, 2]
        * gt_boxes[:, 3]
    )[None, :]

    union = (
        track_area
        + gt_area
        - intersection
    )

    iou = np.divide(
        intersection,
        union,
        out=np.zeros_like(
            intersection
        ),
        where=union > 0,
    )

    return iou


# ============================================================
# HUNGARIAN MATCHING
# ============================================================

matches = []

for frame_id, frame_tracks in tracks.groupby(
    "frame_id"
):

    frame_gt = gt[
        gt["frame_id"] == frame_id
    ]

    if frame_gt.empty:
        continue

    track_boxes = (
        frame_tracks[
            ["x", "y", "w", "h"]
        ]
        .to_numpy()
    )

    gt_boxes = (
        frame_gt[
            ["x", "y", "w", "h"]
        ]
        .to_numpy()
    )

    iou_matrix = bbox_iou_matrix(
        track_boxes,
        gt_boxes,
    )

    if iou_matrix.size == 0:
        continue

    # Hungarian minimizes cost.
    # Therefore use 1 - IoU.
    cost_matrix = (
        1.0
        - iou_matrix
    )

    row_indices, col_indices = (
        linear_sum_assignment(
            cost_matrix
        )
    )

    for row, col in zip(
        row_indices,
        col_indices,
    ):

        iou = iou_matrix[
            row,
            col
        ]

        if iou < IOU_THRESHOLD:
            continue

        track_row = (
            frame_tracks.iloc[row]
        )

        gt_row = (
            frame_gt.iloc[col]
        )

        matches.append({
            "frame_id": int(
                frame_id
            ),

            "mixsort_id": int(
                track_row["mixsort_id"]
            ),

            "canonical_id": int(
                track_row["canonical_id"]
            ),

            "gt_id": int(
                gt_row["gt_id"]
            ),

            "iou": float(iou),
        })


matches_df = pd.DataFrame(
    matches
)

matches_df.to_csv(
    MATCH_LOG_PATH,
    index=False,
)

# ============================================================
# BASIC RESULTS
# ============================================================

print("\n" + "=" * 80)
print("ONE-TO-ONE MATCHING RESULTS")
print("=" * 80)

print(
    f"\nMatched observations: "
    f"{len(matches_df):,}"
)

print(
    f"Unique MixSort IDs matched: "
    f"{matches_df['mixsort_id'].nunique()}"
)

print(
    f"Unique GT players matched: "
    f"{matches_df['gt_id'].nunique()}"
)

print(
    f"Mean IoU: "
    f"{matches_df['iou'].mean():.4f}"
)

# ============================================================
# MIXSORT ID → GT ID DISTRIBUTION
# ============================================================

mixsort_gt = (
    matches_df
    .groupby(
        "mixsort_id"
    )
    .agg(
        gt_ids=(
            "gt_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        gt_id_count=(
            "gt_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

# ============================================================
# GT ID → MIXSORT FRAGMENTS
# ============================================================

gt_mixsort = (
    matches_df
    .groupby(
        "gt_id"
    )
    .agg(
        mixsort_ids=(
            "mixsort_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        mixsort_id_count=(
            "mixsort_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

# ============================================================
# TRUE MIXSORT ID SWITCH CANDIDATES
# ============================================================

contaminated = mixsort_gt[
    mixsort_gt[
        "gt_id_count"
    ] > 1
].copy()

contaminated = contaminated.sort_values(
    [
        "gt_id_count",
        "matched_frames",
    ],
    ascending=False,
)

print("\n" + "=" * 80)
print("ONE-TO-ONE MIXSORT IDs MATCHED TO MULTIPLE GT PLAYERS")
print("=" * 80)

print(
    f"Count: {len(contaminated)}"
)

display(
    contaminated
)

# ============================================================
# CLEAN MIXSORT TRACKS
# ============================================================

clean_mixsort = mixsort_gt[
    mixsort_gt[
        "gt_id_count"
    ] == 1
].copy()

print("\n" + "=" * 80)
print("CLEAN MIXSORT IDs")
print("=" * 80)

print(
    f"Count: {len(clean_mixsort)}"
)

display(
    clean_mixsort.sort_values(
        "matched_frames",
        ascending=False,
    )
)

# ============================================================
# MOST FRAGMENTED GT PLAYERS
# ============================================================

fragmented_gt = gt_mixsort[
    gt_mixsort[
        "mixsort_id_count"
    ] > 1
].copy()

fragmented_gt = fragmented_gt.sort_values(
    [
        "mixsort_id_count",
        "matched_frames",
    ],
    ascending=False,
)

print("\n" + "=" * 80)
print("GT PLAYERS WITH MULTIPLE MIXSORT IDs")
print("=" * 80)

print(
    f"Count: {len(fragmented_gt)}"
)

display(
    fragmented_gt
)

# ============================================================
# GT FRAGMENTATION DISTRIBUTION
# ============================================================

fragmentation_distribution = (
    gt_mixsort[
        "mixsort_id_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "mixsort_ids_per_gt"
    )
    .reset_index(
        name="gt_player_count"
    )
)

print("\n" + "=" * 80)
print("GT FRAGMENTATION DISTRIBUTION")
print("=" * 80)

display(
    fragmentation_distribution
)

# ============================================================
# TEMPORAL ID SWITCH EVENTS
# ============================================================

# For every MixSort ID, inspect chronological GT assignment.
switch_events = []

for mixsort_id, group in matches_df.groupby(
    "mixsort_id"
):

    group = group.sort_values(
        "frame_id"
    )

    previous_gt = None
    previous_frame = None

    for _, row in group.iterrows():

        current_gt = int(
            row["gt_id"]
        )

        current_frame = int(
            row["frame_id"]
        )

        if (
            previous_gt is not None
            and current_gt != previous_gt
        ):

            switch_events.append({
                "mixsort_id": int(
                    mixsort_id
                ),

                "from_gt_id": previous_gt,

                "to_gt_id": current_gt,

                "switch_frame": current_frame,

                "previous_frame": previous_frame,

                "frame_gap": (
                    current_frame
                    - previous_frame
                ),

                "current_iou": float(
                    row["iou"]
                ),
            })

        previous_gt = current_gt
        previous_frame = current_frame


switch_df = pd.DataFrame(
    switch_events
)

switch_df.to_csv(
    SWITCH_LOG_PATH,
    index=False,
)

print("\n" + "=" * 80)
print("TEMPORAL MIXSORT ID SWITCH EVENTS")
print("=" * 80)

print(
    f"Switch events: {len(switch_df)}"
)

if len(switch_df) > 0:

    display(
        switch_df.sort_values(
            "switch_frame"
        )
    )

# ============================================================
# CANONICAL ID → GT ANALYSIS
# ============================================================

canonical_gt = (
    matches_df
    .groupby(
        "canonical_id"
    )
    .agg(
        gt_ids=(
            "gt_id",
            lambda x: sorted(
                x.unique()
            ),
        ),

        gt_id_count=(
            "gt_id",
            "nunique",
        ),

        mixsort_ids=(
            "mixsort_id",
            "nunique",
        ),

        matched_frames=(
            "frame_id",
            "nunique",
        ),

        mean_iou=(
            "iou",
            "mean",
        ),
    )
    .reset_index()
)

canonical_contaminated = canonical_gt[
    canonical_gt[
        "gt_id_count"
    ] > 1
].sort_values(
    "gt_id_count",
    ascending=False,
)

print("\n" + "=" * 80)
print("CANONICAL IDs MATCHED TO MULTIPLE GT PLAYERS")
print("=" * 80)

print(
    f"Count: "
    f"{len(canonical_contaminated)}"
)

display(
    canonical_contaminated
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL ONE-TO-ONE GT ANALYSIS")
print("=" * 80)

print(
    f"GT players:                    "
    f"{gt['gt_id'].nunique()}"
)

print(
    f"MixSort IDs:                   "
    f"{tracks['mixsort_id'].nunique()}"
)

print(
    f"Canonical IDs:                 "
    f"{tracks['canonical_id'].nunique()}"
)

print(
    f"Matched observations:          "
    f"{len(matches_df):,}"
)

print(
    f"Mean IoU:                      "
    f"{matches_df['iou'].mean():.4f}"
)

print(
    f"MixSort IDs → multiple GT:     "
    f"{len(contaminated)}"
)

print(
    f"GT players → multiple MixSort: "
    f"{len(fragmented_gt)}"
)

print(
    f"Temporal switch events:        "
    f"{len(switch_df)}"
)

print(
    f"Canonical IDs → multiple GT:   "
    f"{len(canonical_contaminated)}"
)

print("\nSaved:")
print(MATCH_LOG_PATH)
print(SWITCH_LOG_PATH)

print("\nDone.")

ONE-TO-ONE HUNGARIAN GT IDENTITY ANALYSIS

GT rows: 6,183
GT players: 20
Track rows: 6,632
MixSort IDs: 42

ONE-TO-ONE MATCHING RESULTS

Matched observations: 5,755
Unique MixSort IDs matched: 37
Unique GT players matched: 20
Mean IoU: 0.8451

ONE-TO-ONE MIXSORT IDs MATCHED TO MULTIPLE GT PLAYERS
Count: 10


,mixsort_id,gt_ids,gt_id_count,matched_frames,mean_iou
20,417,"[2, 5, 15, 18]",4,204,0.840010
21,419,"[1, 2, 7, 15]",4,183,0.854839
16,405,"[3, 5, 14, 15]",4,6,0.623625
15,403,"[3, 4, 18]",3,430,0.823029
6,394,"[7, 13, 17]",3,375,0.854671
18,407,"[3, 6, 14]",3,358,0.835765
10,398,"[5, 7]",2,386,0.836406
17,406,"[5, 18]",2,340,0.838475
34,574,"[4, 18]",2,50,0.787263
33,571,"[8, 12]",2,45,0.837967



CLEAN MIXSORT IDs
Count: 27


,mixsort_id,gt_ids,gt_id_count,matched_frames,mean_iou
0,387,[10],1,429,0.857954
11,399,[13],1,380,0.856482
12,400,[4],1,364,0.843474
3,391,[16],1,251,0.880825
9,397,[12],1,239,0.873129
14,402,[8],1,209,0.845831
8,396,[6],1,174,0.876627
19,416,[2],1,169,0.822477
7,395,[3],1,169,0.843166
23,423,[19],1,131,0.836141



GT PLAYERS WITH MULTIPLE MIXSORT IDs
Count: 16


...


TEMPORAL MIXSORT ID SWITCH EVENTS
Switch events: 19


,mixsort_id,from_gt_id,to_gt_id,switch_frame,previous_frame,frame_gap,current_iou
5,405,5,15,243,179,64,0.511172
14,419,1,15,373,372,1,0.649905
9,407,14,3,377,372,5,0.610107
3,403,18,4,377,376,1,0.549061
11,417,15,2,377,372,5,0.724983
6,405,15,14,378,243,135,0.819199
8,406,5,18,379,368,11,0.749728
0,394,17,13,380,372,8,0.557589
15,419,15,2,380,376,4,0.793125
7,405,14,3,382,378,4,0.752908



CANONICAL IDs MATCHED TO MULTIPLE GT PLAYERS
Count: 10


,canonical_id,gt_ids,gt_id_count,mixsort_ids,matched_frames,mean_iou
20,23,"[2, 5, 15, 18]",4,1,204,0.840010
16,19,"[3, 5, 14, 15]",4,1,6,0.623625
21,24,"[1, 2, 7, 15]",4,2,191,0.849366
6,8,"[7, 13, 17]",3,1,375,0.854671
18,21,"[3, 6, 14]",3,1,358,0.835765
15,17,"[3, 4, 18]",3,1,430,0.823029
10,12,"[5, 7]",2,1,386,0.836406
17,20,"[5, 18]",2,1,340,0.838475
32,38,"[8, 12]",2,1,45,0.837967
33,39,"[4, 18]",2,1,50,0.787263



FINAL ONE-TO-ONE GT ANALYSIS
GT players:                    20
MixSort IDs:                   42
Canonical IDs:                 41
Matched observations:          5,755
Mean IoU:                      0.8451
MixSort IDs → multiple GT:     10
GT players → multiple MixSort: 16
Temporal switch events:        19
Canonical IDs → multiple GT:   10

Saved:
/mnt/d/My Projects/AthletiQ/AthletiQ/outputs/tracking/c017_hungarian_gt_matches_v1.csv
/mnt/d/My Projects/AthletiQ/AthletiQ/outputs/tracking/c017_mixsort_switch_events_v1.csv

Done.
